# Sesión 6 — Hive Metastore, tablas Hive y Trino

**En clase seguiremos la misma venta desde su fichero Parquet hasta una consulta por nombre en Spark y Trino.**

Esta sesión viene después de S5. Ya has visto que una colección
de Parquet puede organizarse en directorios particionados y que Spark puede
leerlos directamente. Ahora se introduce la capa que proporciona nombres y
metadatos compartidos a los motores.

La pregunta central es:

> ¿Cómo pasamos de conocer una ruta física a disponer de una tabla lógica que
> puedan consultar Spark y Trino?

El Hive Metastore no es un motor de procesamiento y Trino no es el catálogo.
El metastore conserva metadatos; Trino consulta y procesa; Spark puede
utilizar la misma definición si se configura como cliente del catálogo.

La figura sitúa la sesión en la arquitectura física del laboratorio: cada círculo es un contenedor Docker, y los puntos de color de su interior son los servicios que ejecuta; a la derecha están los servicios, con el color de la sesión que los introduce y una flecha hacia el clúster. En color está lo que ya existe al empezar la sesión y sombreado, lo que llegará en sesiones posteriores; resaltado, lo que esta sesión crea, modifica o usa. S6 añade el Hive Metastore (con su PostgreSQL) y Trino: Spark y Trino comparten desde ahora las definiciones de tabla sobre los ficheros del data lake.

<p align="center" style="text-align: center;"><img src="https://dsevilla.github.io/tcdm-public/figs/arquitectura-s6.svg" alt="Arquitectura del laboratorio al empezar la sesión 6: se añaden el Hive Metastore con PostgreSQL y Trino" width="1100" style="display: block; margin: 0 auto; max-width: 100%; height: auto;"></p>

La infraestructura definida en `entorno/compose-warehouse-hdfs.yml` separa
PostgreSQL, Hive Metastore, Trino y el clúster Hadoop. PostgreSQL almacena el
repositorio del metastore; Hive Metastore expone la información por Thrift;
Trino utiliza esa información para planificar consultas. Ninguno de esos
servicios se convierte por ello en DataNode o NodeManager.

> **Paquetes de Python de esta sesión.** S6 usa `pyspark` para Spark,
> `trino` para enviar consultas al servidor y `pandas` para organizar los
> resultados. Todos los notebooks de S2-S8 instalan la lista común publicada
> como `entorno/requirements.txt`; allí están reunidos los paquetes usados a
> lo largo del curso.

## Actualizar el material antes de empezar

Antes de empezar, desde la raíz de la distribución y en una terminal de tu
equipo:

```bash
make -C entorno actualizar
```

La orden trae la última versión publicada del material. No pisa tu trabajo: si
has modificado un fichero que también cambia en la versión nueva, se aparta
como copia `<fichero>.mio-<fecha>-<hora>` y el original pasa a ser el
publicado. Lo que la versión nueva no cambia se queda como está.


### Consultar el esquema de TPC-DS

Como en S2, `%esquema` muestra debajo de la celda en la que lo escribas la
descripción de una tabla o de una columna de TPC-DS, tomada del apéndice de
`s2.ipynb`:

| Celda | Qué muestra |
| --- | --- |
| `%esquema` | Las 18 tablas, con su papel y su descripción |
| `%esquema web_sales` | La tabla completa: su descripción, su diagrama entidad-relación si es una tabla de hechos, y todas sus columnas |
| `%esquema web_sales date` | Las columnas de `web_sales` cuyo nombre contiene `date` |
| `%esquema ws_sold_timestamp` | Esa columna y ninguna otra |

Una celda `%%esquema` admite una tabla, columna o filtro por línea. Lo
encontrarás la primera vez que la sesión usa cada tabla; añade una celda de
éstas donde te haga falta y bórrala después.

`%schema` y `%%schema` son los mismos comandos con su nombre en inglés:
usa el que prefieras.

La celda siguiente lo prepara: ejecútala una vez por sesión de kernel.
Descarga `entorno/python_utils/tcdm_esquema.py` del repositorio público
`dsevilla/tcdm-public` (rama `26-27`), porque el kernel se ejecuta en
`namenode`, que no ve tu copia de la distribución. Si el fichero ya está en
el directorio de trabajo del kernel, se conserva; bórralo para traer una
versión nueva.

In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen

SCHEMA_MODULE: Path = Path("tcdm_esquema.py")
schema_module_url: str = (
    "https://raw.githubusercontent.com/dsevilla/tcdm-public/26-27/"
    "entorno/python_utils/tcdm_esquema.py"
)

if not SCHEMA_MODULE.exists():
    request: Request = Request(schema_module_url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(request, timeout=30) as response:
        SCHEMA_MODULE.write_bytes(response.read())

%load_ext tcdm_esquema

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:linear-gradient(135deg,#0c304d,#174f7a 62%,#a54467);color:#ffffff;border:1px solid transparent;display:flex;flex-direction:column;justify-content:center;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="margin:0 auto;max-width:44ch;text-align:left"><h1 align="center" style="color:#ffffff;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;text-align:center;margin:0 0 22px">Sesión 6: Hive Metastore, tablas Hive y Trino</h1><h2 align="center" style="color:#dceaf4;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:30px;text-align:center;margin:18px 0 8px">De la ruta física a la tabla lógica</h2><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Un metastore, dos motores: Trino y Spark leen la misma definición</li><li style='margin:10px 0;list-style-position:outside'>Tabla externa frente a tabla administrada</li><li style='margin:10px 0;list-style-position:outside'>El recuento cambia de 0 a N tras sincronizar las particiones</li></ul></div><hr color="#174f7a" size="8" noshade style="display:none"></div>

## Capas que se deben distinguir

| Capa | Responsabilidad | Ejemplo del laboratorio |
| --- | --- | --- |
| Almacenamiento | Conserva bytes y rutas | HDFS bajo `/datalake` o `/warehouse` |
| Formato | Describe cómo leer cada fichero | Parquet |
| Metastore | Guarda nombres, esquemas, ubicaciones y particiones | Hive Metastore + PostgreSQL |
| Motor SQL | Planifica y ejecuta consultas | Trino |
| Motor de transformación | Produce o consume DataFrames | Spark |
| Formato de tabla | Gestiona metadatos de tabla y cambios | Iceberg en S7 |

Una ruta Parquet puede existir sin metastore. Una entrada del metastore no
contiene los bytes de la tabla. Una tabla Hive externa puede apuntar a
datos existentes y no debe borrar esos datos automáticamente al eliminar sólo
su definición. Esa distinción entre "borrar metadatos" y "borrar datos" es
uno de los hilos conductores de esta sesión.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Del fichero a la tabla</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Seguiremos una venta desde su directorio en HDFS hasta su nombre en el catálogo.</li><li style='margin:10px 0;list-style-position:outside'>Al final compararemos lo que leen Spark y Trino.</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Objetivos de la sesión

Al terminar esta sesión deberías poder:

- explicar qué problema resuelve un catálogo frente a una ruta física,
  también cuando una tabla tiene miles de particiones;
- distinguir Hive Metastore, Trino, Spark y PostgreSQL en esta arquitectura;
- registrar Parquet ya existente como tabla externa, sin copiar ni
  transformar los datos;
- distinguir una tabla externa de una tabla administrada, saber en qué
  ubicación física vive cada una y qué ventaja da que una tabla tenga un
  único dueño;
- consultar la ubicación, el esquema y las propiedades de una tabla con
  `SHOW CREATE TABLE` y `DESCRIBE`;
- inspeccionar las particiones que conoce el metastore de una tabla Hive
  particionada, y entender por qué hace falta un paso explícito de
  descubrimiento tras el `CREATE TABLE`;
- ejecutar consultas equivalentes desde Trino y desde Spark contra la misma
  definición de tabla, sin tratarlos como catálogos distintos;
- observar selección de columnas, *predicate pushdown* y eliminación de
  particiones en el plan de consulta, no sólo en el tiempo de pared;
- calcular estadísticas con `ANALYZE` y explicar para qué sirven;
- documentar el ciclo de vida de una tabla y el riesgo de borrar datos
  gestionados por accidente;
- apuntar en el catálogo de qué se calcula cada tabla (su linaje) y usarlo
  para saber qué recalcular, y en qué orden, cuando cambia una fuente;
- registrar con Spark todas las tablas de `raw`, comprobar que Trino las lee
  y explicar cuándo una tabla registrada por un motor no la entiende el otro;
- buscar en el catálogo por nombre, por comentario y por ubicación, con
  `information_schema` en Trino y con un inventario en Spark;
- identificar las limitaciones del Hive Metastore que justifican introducir
  Iceberg en S7, y explicar por qué Iceberg se construye sobre una tabla
  administrada del warehouse.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #b3701a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#b3701a" size="8" noshade style="display:none"><div style="color:#b3701a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#b3701a" style="color:inherit"><b>Evaluación de la sesión</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Cómo se evalúa esta sesión</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Reunión individual breve con el profesor</li><li style='margin:10px 0;list-style-position:outside'>Se muestra en vivo: los <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>SHOW CREATE TABLE</code> externos, el recuento 0 → N tras <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sync_partition_metadata</code>, y la comparación de negocio entre Trino y Spark sobre <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>gold</code></li><li style='margin:10px 0;list-style-position:outside'>Se entrega también una memoria breve (una o dos páginas)</li></ul><hr color="#b3701a" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Preparar las dos conexiones</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Localiza silver y gold en HDFS y comprueba que el warehouse está disponible.</li><li style='margin:10px 0;list-style-position:outside'>Usaremos esos mismos datos durante toda la sesión.</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Antes de empezar

> **Dónde se ejecuta este notebook.** Igual que en S2, S3, S4 y S5, Jupyter
> se ejecuta directamente dentro del contenedor `namenode`, como `luser`, con
> acceso de red directo a HDFS, YARN, Trino, PostgreSQL y el Hive Metastore:
> todos comparten la misma red Docker `hadoop-cluster`. Las celdas
> ejecutables hablan directamente con esos servicios. Las órdenes que
> necesitan el Docker del host — arrancar o detener contenedores, `make` — se
> muestran como texto para ejecutarlas en una terminal de tu equipo, nunca
> como celdas de este notebook.

Esta sesión asume que:

- el clúster de S1 ya está en marcha;
- S2 generó los Parquet de TPC-DS SF3 en `/datalake/raw/tpcds`;
- S5 ya ejecutó su ETL y dejó `/datalake/silver/tpcds/web_sales` (particionado
  por `sold_year` y `sold_month`) y `/datalake/gold/tpcds/web_sales_by_year`
  escritos en HDFS.

Si el clúster de infraestructura (PostgreSQL, Hive Metastore y Trino) no está
levantado todavía, desde una terminal de tu equipo situada en la raíz de la
distribución de sesiones:

<p align="center" style="text-align: center;"><img src="https://dsevilla.github.io/tcdm-public/figs/anim-host-warehouse-up.gif" alt="Animación: la orden make -C entorno warehouse-up se teclea en una terminal de tu equipo (host)" width="832" style="display: block; margin: 0 auto; max-width: 100%; height: auto;"></p>

```bash
cd ~/tcdm-public
make -C entorno warehouse-up
```

`warehouse-up` depende de `hadoop-up`, así que también arranca el clúster
Hadoop si hiciera falta, y arranca además el servicio `catalog-init`, que
crea el esquema `hive.tcdm` la primera vez que se levanta el entorno.

Este notebook se ejecuta en el Jupyter de `namenode`, que no arranca solo.
Como en S2, arráncalo desde **otra terminal de tu equipo** y déjala abierta
mientras trabajes:

```bash
# en tu equipo: entra en namenode
docker exec -it namenode bash
# ya dentro de namenode, como luser: Jupyter con el token fijo «tcdm»
su - luser
jupyter lab --ip=0.0.0.0 --port=8888 --no-browser --IdentityProvider.token=tcdm
```

La primera orden es la única que se ejecuta en tu equipo; las otras dos, ya
dentro del contenedor, arrancan Jupyter con el Python del curso
(`make -C entorno jupyter` hace lo mismo en una sola orden). En Visual
Studio Code, conecta este notebook con **Select Kernel → Select Another
Kernel → Existing Jupyter Server**, usando la dirección
`http://127.0.0.1:8888/lab?token=tcdm` y eligiendo el kernel **Python 3
(ipykernel)**. El token es siempre `tcdm`.

Si algo no funciona —el notebook no conecta, una celda `!hdfs` no
encuentra la orden, un servicio no responde—, consulta «Solución de
problemas» en `entorno/README.md`.

Esta sesión no regenera S5. Si `/datalake/silver/tpcds/web_sales` o
`/datalake/gold/tpcds/web_sales_by_year` no existen todavía, ejecuta antes el
notebook de S5.

In [ ]:
!hdfs dfs -ls -d /datalake/raw/tpcds/item \
    /datalake/silver/tpcds/web_sales \
    /datalake/gold/tpcds/web_sales_by_year \
    /warehouse

Si alguna de esas cuatro rutas no aparece, revisa S2 (para `item`)
o S5 (para `web_sales` en `silver` y `web_sales_by_year` en `gold`) antes de
continuar. `/warehouse` ya existe desde que S1 preparó el NameNode: es la
zona reservada para las tablas administradas por Hive e Iceberg, y hasta
ahora sólo contiene el directorio vacío `tcdm.db`, que se creó al arrancar
el warehouse.

## Instalar las dependencias de este notebook

El notebook usa PySpark, `trino` —el cliente DB-API 2.0 para enviar
consultas al servidor Trino— y `pandas` para organizar los resultados.

La distribución publica una sola lista común en `entorno/requirements.txt`.
La celda siguiente la instala directamente con `%pip` desde la URL pública,
que aparece en la celda de código. La lista cubre las dependencias de todas
las sesiones S2-S8; la imagen de `namenode` ya trae esos paquetes en
`/opt/tcdm/venv`, por lo que lo normal es que `%pip` confirme que están.

La celda siguiente instala la lista común publicada en `entorno/requirements.txt`, directamente desde su URL HTTPS. `pip` admite una URL en `-r`, así que no hace falta escribir una copia local del fichero.

In [ ]:
%pip install -q -r https://raw.githubusercontent.com/dsevilla/tcdm-public/26-27/entorno/requirements.txt

In [ ]:
import sys

import pandas
import pyspark
import trino

print(f"PySpark {pyspark.__version__} con Python {sys.version.split()[0]}.")
print(f"Cliente trino: {trino.__version__}")
print(f"Pandas: {pandas.__version__}")

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Dos clientes del mismo metastore</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>enableHiveSupport()</code>, la URI Thrift del Hive Metastore y su cliente Hive 4.1.0</li><li style='margin:10px 0;list-style-position:outside'>El cliente Python <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>trino</code> habla con Trino sobre la misma red Docker</li><li style='margin:10px 0;list-style-position:outside'>Sin CLI de Trino ni <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>docker exec</code> desde el kernel</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Construir la SparkSession con soporte Hive

A diferencia de S4 —donde Spark leía Parquet directamente por ruta y no
necesitaba ningún catálogo—, aquí Spark debe hablar con el **mismo** Hive
Metastore que usa Trino. Eso exige cuatro configuraciones explícitas, además
de `enableHiveSupport()`. Las dos primeras dicen dónde está el metastore y
dónde están los datos:

- `spark.hadoop.hive.metastore.uris`: la URI Thrift del metastore,
  `thrift://hive-metastore:9083`. Es el mismo puerto que declara el
  *healthcheck* del servicio `hive-metastore` en
  `entorno/compose-warehouse-hdfs.yml`.
- `spark.sql.warehouse.dir`: la ubicación por defecto de las tablas
  administradas, `hdfs://namenode:9000/warehouse`. Coincide con
  `metastore.warehouse.dir`, que ese mismo entorno fija en
  `entorno/hive-metastore/conf/metastore-site.xml` para `hive-metastore`,
  para que Spark y el metastore elijan la misma ruta si alguno crea una tabla
  administrada sin indicar `LOCATION`.

Las otras dos no tienen que ver con dónde está el metastore, sino con **qué
versión** es. El curso usa Hive Metastore 4.2.1.
forma aislada, el cliente de otra versión:

- `spark.sql.hive.metastore.version=4.1.0`: la versión del cliente Hive con
  la que Spark habla con el metastore. Es la más reciente que admite Spark
  4.1 y entiende el protocolo del servidor 4.2.1.
- `spark.sql.hive.metastore.jars`: de dónde saca Spark los jars de ese
  cliente, unos 300 MB porque arrastran buena parte de Hive.

Como en la parte distribuida de S4, el driver vive en `namenode` y YARN
asigna los ejecutores a los NodeManagers. La celda siguiente cuenta los que
están `RUNNING` y solicita, por defecto, un ejecutor por cada uno; YARN decide
su colocación, por lo que no se garantiza uno en cada DataNode. Para pedir
otra cantidad, fija antes `%env TCDM_SPARK_EXECUTORS=5` en una celda del
notebook, como en S4. Cierra cualquier otra sesión Spark antes de continuar
para que no retenga recursos ni ocupe el puerto 4040.

Cada ejecutor pide un núcleo, 512 MiB de memoria y 256 MiB adicionales;
el ApplicationMaster pide 512 MiB en total. Cada NodeManager anuncia
2560 MiB. La sesión y sus contenedores se liberan con `spark.stop()`.
Si faltan NodeManagers, comprueba el clúster desde una terminal de tu equipo
o en el ResourceManager (`http://localhost:8088`).

In [ ]:
import os
import subprocess

assert os.environ.get("HADOOP_CONF_DIR"), (
    "El kernel no hereda HADOOP_CONF_DIR: arranca Jupyter como luser dentro "
    "de namenode, como se explica en «Antes de empezar»."
)
node_result: subprocess.CompletedProcess[str] = subprocess.run(
    ["yarn", "node", "-list", "-all"], capture_output=True, text=True, check=True
)
running_nodemanagers: int = sum(
    1
    for line in node_result.stdout.splitlines()
    if len(line.split()) >= 2 and line.split()[1] == "RUNNING"
)
print(f"NodeManagers en estado RUNNING: {running_nodemanagers}")
assert running_nodemanagers >= 2, (
    "Se esperan al menos dos NodeManagers RUNNING. Comprueba los DataNodes "
    "con `make -C entorno status` en una terminal de tu equipo o en "
    "http://localhost:8088; si acabas de arrancar el clúster, espera y repite."
)
requested_executors: int = int(os.environ.get("TCDM_SPARK_EXECUTORS", running_nodemanagers))
assert requested_executors > 0, "TCDM_SPARK_EXECUTORS debe ser un entero positivo."
print(f"Solicitando {requested_executors} ejecutores Spark a YARN.")

In [ ]:
from pathlib import Path

from pyspark import SparkContext
from pyspark.sql import SparkSession

HIVE_METASTORE_URIS: str = "thrift://hive-metastore:9083"
WAREHOUSE_DIR: str = "hdfs://namenode:9000/warehouse"
HIVE_CLIENT_VERSION: str = "4.1.0"
# La imagen de namenode trae el cliente Hive en este directorio. Con una
# imagen anterior que no lo tenga, Spark lo descarga de Maven Central.
HIVE_CLIENT_DIR: Path = Path("/opt/tcdm/hive-metastore-client") / HIVE_CLIENT_VERSION
HIVE_CLIENT_CONF: dict[str, str] = (
    {
        "spark.sql.hive.metastore.jars": "path",
        "spark.sql.hive.metastore.jars.path": f"file://{HIVE_CLIENT_DIR}/*.jar",
    }
    if any(HIVE_CLIENT_DIR.glob("*.jar"))
    else {"spark.sql.hive.metastore.jars": "maven"}
)

spark: SparkSession = (
    SparkSession.builder.appName("TCDM_S6_YARN")
    .master("yarn")
    .config("spark.executor.instances", str(requested_executors))
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "512m")
    .config("spark.executor.memoryOverhead", "256m")
    .config("spark.yarn.am.cores", "1")
    .config("spark.yarn.am.memory", "256m")
    .config("spark.yarn.am.memoryOverhead", "256m")
    .config("spark.scheduler.minRegisteredResourcesRatio", "1.0")
    .config("spark.scheduler.maxRegisteredResourcesWaitingTime", "60s")
    .config("spark.pyspark.python", "python3")
    .config("spark.yarn.isPython", "true")
    .config("spark.driver.memory", "1g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.hadoop.hive.metastore.uris", HIVE_METASTORE_URIS)
    .config("spark.sql.warehouse.dir", WAREHOUSE_DIR)
    .config("spark.sql.catalogImplementation", "hive")
    .config("spark.sql.hive.metastore.version", HIVE_CLIENT_VERSION)
    .config(map=HIVE_CLIENT_CONF)
    .enableHiveSupport()
    .getOrCreate()
)
sc: SparkContext = spark.sparkContext
assert sc.master == "yarn", (
    "Ya hay una sesión Spark con otro master. Ciérrala con spark.stop() "
    "y vuelve a ejecutar esta celda."
)

print(f"Spark version: {spark.version} | Master: {sc.master}")
print(f"Metastore URIs: {HIVE_METASTORE_URIS}")
print(
    f"Cliente Hive {HIVE_CLIENT_VERSION}: jars={HIVE_CLIENT_CONF['spark.sql.hive.metastore.jars']}"
)
print(f"Warehouse dir: {WAREHOUSE_DIR}")

`spark.sql.catalogImplementation=hive` junto con `enableHiveSupport()`
son lo que hace que Spark deje de usar su catálogo en memoria (el que veías
en S4 al registrar vistas temporales con `createOrReplaceTempView`) y se
convierta en **otro cliente** del mismo Hive Metastore que usará Trino. Una
comprobación rápida: si no aparece `tcdm` entre las bases de datos, esta
`SparkSession` no está viendo el mismo metastore que Trino; si sólo
apareciera `default` (la base heredada de Hive, siempre presente),
estaríamos ante el catálogo en memoria de siempre, no ante el Hive
Metastore.

In [ ]:
spark.sql("SHOW DATABASES").show()

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Qué es Trino</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Un motor SQL que no guarda datos propios: los lee de otros sistemas a través de conectores</li><li style='margin:10px 0;list-style-position:outside'>Un <em>coordinator</em> planifica y unos <em>workers</em> ejecutan; aquí, un único contenedor hace las dos cosas</li><li style='margin:10px 0;list-style-position:outside'>El conector <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>hive</code> declara tablas sobre ficheros de HDFS y guarda su definición en Hive Metastore</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Qué es Trino

[Trino](https://trino.io/) (antes PrestoSQL) es un motor de consultas SQL
distribuido para *federar* datos: no almacena sus propias tablas, sino que
se conecta a orígenes heterogéneos a través de *conectores* —HDFS/Hive,
Iceberg, PostgreSQL, Kafka, entre
[muchos otros](https://trino.io/docs/current/connector.html)— y los expone
bajo el mismo dialecto SQL ANSI. Una sola consulta puede combinar en el
mismo `JOIN` tablas que viven en conectores distintos.

Internamente separa dos papeles: un *coordinator*, que analiza y planifica
cada consulta, y uno o varios *workers*, que ejecutan las tareas en paralelo
(arquitectura MPP, *massively parallel processing*). El laboratorio
despliega un único contenedor, `trino-hdfs`, que asume los dos papeles,
suficiente para SF3; en producción, coordinator y workers escalan por
separado.

Cada conector se activa con un fichero de propiedades en
`entorno/trino-hdfs/etc/catalog/`, y el nombre del fichero es el nombre del
catálogo en SQL: `hive.properties` da el catálogo `hive` de las consultas
`hive.tcdm.<tabla>`. Esta sesión usa el conector **`hive`**, que declara
tablas sobre ficheros de HDFS (aquí, Parquet) y guarda su definición en
Hive Metastore; S7 añadirá el conector `iceberg`, que habla con el mismo
metastore a través de su catálogo Iceberg REST.

```properties
# hive.properties
connector.name=hive
hive.metastore=thrift
hive.metastore.uri=thrift://hive-metastore:9083
hive.metastore.authentication.type=NONE
hive.metastore.thrift.impersonation.enabled=false
hive.hdfs.authentication.type=NONE
hive.hdfs.impersonation.enabled=false
hive.non-managed-table-creates-enabled=true
hive.non-managed-table-writes-enabled=true
hive.storage-format=PARQUET
hive.compression-codec=ZSTD
fs.hadoop.enabled=true
parquet.writer.row-group-size=32MB
```

- `hive.metastore.uri` dice dónde está el metastore: el mismo servicio
  Thrift al que se ha conectado Spark más arriba.
- Las propiedades `*.authentication.type=NONE` y
  `*.impersonation.enabled=false` desactivan Kerberos y la suplantación de
  usuario: los accesos de Trino a HDFS y al metastore usan la identidad
  `luser`, sin un usuario intermedio que autentique en su nombre.
- Las dos propiedades `hive.non-managed-table-*-enabled=true` permiten crear
  tablas **externas** —las que apuntan a una ruta que ya existe, con
  `external_location`— y escribir en ellas. Sin ellas, los `CREATE TABLE`
  sobre `/datalake` de esta sesión fallarían.
- `hive.storage-format` y `hive.compression-codec` son el formato y la
  compresión de las tablas que escribe el propio Trino, como la tabla
  administrada de más abajo, y `parquet.writer.row-group-size` limita sus
  grupos de filas a 32 MB, la mitad de un bloque de HDFS.

Trino se despliega como un servicio Docker más, definido en
`entorno/compose-warehouse-hdfs.yml`:

```yaml
trino-hdfs:
  image: trinodb/trino:${TRINO_VERSION:-483}
  depends_on:
    hive-metastore:
      condition: service_healthy
  volumes:
    - ./trino-hdfs/etc:/etc/trino:ro
  networks:
    hadoop-cluster:
```

El volumen monta ese directorio de configuración como `/etc/trino` dentro
del contenedor, y `depends_on` obliga a esperar a que Hive Metastore
responda antes de arrancar, porque el conector `hive` lo necesita para
resolver esquemas y tablas. Trino no guarda nada por su cuenta: para
resolver `hive.tcdm.item` pregunta al metastore qué columnas tiene esa
tabla y en qué ruta de HDFS están sus ficheros, y luego lee los ficheros
directamente de HDFS.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Probar Trino</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Desde el kernel de namenode enviaremos una consulta SQL al servicio trino-hdfs.</li><li style='margin:10px 0;list-style-position:outside'>Un SELECT 1 confirma la conexión antes de acceder al catálogo.</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Abrir la conexión con Trino

Usamos el paquete `trino` (PyPI) en lugar de la CLI: `trino.dbapi.connect`
habla el protocolo HTTP de Trino directamente contra el servicio
`trino-hdfs` de la red `hadoop-cluster`, en su puerto interno `8080`. Compose
publica ese servicio en el equipo como `127.0.0.1:18080`; el kernel no usa
ese puerto publicado porque ya vive dentro de la red Docker y se conecta al
nombre del servicio. Definimos una pequeña función auxiliar que ejecuta SQL y devuelve un
`DataFrame` de pandas, para no repetir en cada celda la apertura de cursor y
la construcción de columnas.

In [ ]:
import pandas as pd
from trino.dbapi import Connection, Cursor

trino_conn: Connection = trino.dbapi.connect(
    host="trino-hdfs",
    port=8080,
    user="luser",
    catalog="hive",
    schema="tcdm",
)


def run_trino(sql: str) -> pd.DataFrame:
    '''Ejecuta una sentencia en Trino y devuelve el resultado como DataFrame.'''
    cursor: Cursor = trino_conn.cursor()
    cursor.execute(sql)
    rows: list[tuple] = cursor.fetchall()
    columns: list[str] = [description[0] for description in cursor.description or []]
    return pd.DataFrame(rows, columns=columns)

In [ ]:
run_trino("SELECT 1 AS ping")

La conexión responde. `catalog="hive"` y `schema="tcdm"` son los
valores por defecto de `trino_conn`: en el resto del notebook escribiremos
igualmente los nombres completos (`hive.tcdm.<tabla>`) en el SQL para que
cada sentencia se entienda sin depender del contexto por defecto de la
conexión.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Para qué sirve un catálogo</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Un sitio donde todo son tablas: <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>SHOW TABLES</code>, no <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>hdfs dfs -ls -R</code></li><li style='margin:10px 0;list-style-position:outside'>Un nombre en lugar de una ruta, el mismo para Trino y Spark: mover los datos no cambia las consultas</li><li style='margin:10px 0;list-style-position:outside'>Declara lo que los ficheros no dicen: columnas de partición, descripción, linaje</li><li style='margin:10px 0;list-style-position:outside'>Particiones en una tabla indexada: con miles, listar directorios en cada consulta no escala</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Para qué sirve un catálogo

En S5 escribiste `silver` y `gold` como directorios de Parquet, y cada
lectura tenía que conocer de antemano su ruta física. Un catálogo pone encima
una capa de tablas: cada una es una fila en una base de datos, con su nombre,
su ubicación, su esquema y sus particiones. Qué se gana con eso:

- **Un sitio donde todo son tablas.** `SHOW TABLES FROM hive.tcdm` responde
  qué datos hay. Sin catálogo, la respuesta es recorrer `/datalake` y
  `/warehouse` con `hdfs dfs -ls -R` y deducir qué directorio es una tabla,
  cuál una partición y cuál un fichero auxiliar como `_SUCCESS`.
- **Un nombre en lugar de una ruta, el mismo para todos los motores.** Trino
  y Spark consultarán `tcdm.web_sales_silver` sin escribir ninguna ruta. Si
  los datos cambian de sitio —otro directorio, o el almacenamiento S3 de la
  sesión 3—, se cambia la ubicación en el catálogo y ninguna consulta cambia;
  y tablas en HDFS y en S3 pueden convivir en el mismo esquema `tcdm`.
- **Lo que los ficheros no dicen.** Un Parquet guarda sus columnas y sus
  tipos, pero no que forma parte de una tabla llamada `web_sales_silver`.
  `sold_year` y `sold_month` ni siquiera están dentro de los ficheros: sólo
  aparecen, como texto, en el nombre de los directorios, y es el catálogo el
  que declara que son columnas enteras. Tampoco guardan una descripción, un
  propietario ni de qué otras tablas se calcularon (su linaje, que apuntarás
  al final de la sesión).
- **Las particiones, en una tabla indexada.** Es la ventaja que más crece con
  el tamaño. Para saber qué particiones existen y cuáles cumplen el filtro de
  una consulta, un motor sin catálogo tiene que listar directorios cada vez:
  en HDFS, una petición al NameNode por directorio; en S3, peticiones `LIST`
  que devuelven como mucho 1.000 claves cada una, una detrás de otra.
  `web_sales_silver` tiene 61 particiones y no se nota. Una tabla de eventos
  particionada por día y por hora durante diez años tiene 87.600: una
  consulta con `WHERE hora = 3` obligaría a listar los 3.650 directorios de
  día antes de leer un solo byte, y sólo se libra la que filtra por la
  primera columna de partición, porque puede construir la ruta. El metastore
  guarda una fila por partición en `PARTITIONS`, en PostgreSQL y con índices,
  y Trino le pide sólo las que cumplen el filtro: una consulta SQL en lugar de
  un recorrido del árbol. Verás esa poda en el `EXPLAIN` filtrado por
  `sold_year`.
- **Estadísticas para el optimizador.** Filas, valores distintos y nulos por
  columna (`ANALYZE`, más adelante): calcularlas recorriendo los ficheros en
  cada consulta costaría tanto como la propia consulta.

Una nota sobre el esquema: el `CREATE TABLE` de Trino que usarás obliga a
repetir las columnas y los tipos que el Parquet ya guarda. Spark no: puede
deducirlos de los propios ficheros al registrar la tabla.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #0c304d;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#0c304d" size="8" noshade style="display:none"><div style="color:#0c304d;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#0c304d" style="color:inherit"><b>Pregunta guía</b></font></div><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Una tabla particionada por <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>dia</code> y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>hora</code> durante diez años tiene 87.600 directorios. Sin catálogo, ¿cuántos hay que listar para planificar <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>WHERE hora = 3</code>? ¿Y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>WHERE dia = DATE &#x27;2025-03-01&#x27;</code>?</li><li style='margin:10px 0;list-style-position:outside'>¿Qué le pregunta Trino al metastore en lugar de listarlos?</li></ul><hr color="#0c304d" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Registrar sin copiar</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Qué guarda el metastore y qué no toca en HDFS</li><li style='margin:10px 0;list-style-position:outside'>Las tres tablas externas sobre <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>/datalake</code></li><li style='margin:10px 0;list-style-position:outside'>Una tabla administrada de laboratorio en <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>/warehouse</code></li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Qué guarda el catálogo y dónde viven las tablas

Antes de registrar ninguna tabla conviene separar dos lugares. Los **datos**
de una tabla Hive son ficheros en HDFS: los Parquet que escribió S5, con sus
directorios de partición. El **catálogo** —el Hive Metastore, que lo guarda
todo en tablas de su base de datos PostgreSQL (`metastore_db`)— sólo contiene
metadatos: el nombre de cada base de datos y de cada tabla, su tipo (externa
o administrada), su ubicación en HDFS y su formato, las columnas con sus
tipos, las columnas de partición y una fila por cada partición que conoce. Ni
una sola fila de `web_sales` entra en PostgreSQL.

La animación sigue, paso a paso, lo que harás más abajo con
`tcdm.web_sales_silver`, y añade al final qué ocurre al borrar una tabla
externa:

<p align="center" style="text-align: center;"><img src="https://dsevilla.github.io/tcdm-public/figs/catalogo-hive-s6.gif" alt="Animación en seis pasos: el catálogo (Hive Metastore con sus tablas DBS, TBLS, SDS, COLUMNS_V2, PARTITION_KEYS y PARTITIONS) frente a los directorios particionados de web_sales en HDFS; CREATE TABLE escribe la definición, count(*) devuelve 0 sin particiones, sync_partition_metadata añade 61, la consulta lee los Parquet y DROP TABLE borra sólo los metadatos" width="1100" style="display: block; margin: 0 auto; max-width: 100%; height: auto;"></p>

1. **Punto de partida.** Los Parquet de S5 ya están en HDFS, bajo
   `/datalake/silver/tpcds/web_sales`, en directorios
   `sold_year=…/sold_month=…`. El catálogo sólo conoce la base de datos
   `tcdm`.
2. **`CREATE TABLE`** con `external_location` y `partitioned_by` escribe en
   el catálogo la tabla y su tipo (`TBLS`), su ubicación y su formato
   (`SDS`), sus columnas (`COLUMNS_V2`) y sus columnas de partición
   (`PARTITION_KEYS`). No lee ni copia ningún fichero de HDFS.
3. **`SELECT count(*)` devuelve 0.** `PARTITIONS` está vacía, así que Trino
   no sabe qué directorios leer, aunque los ficheros estén ahí.
4. **`hive.system.sync_partition_metadata`** recorre los directorios bajo la
   ubicación de la tabla y añade a `PARTITIONS` una fila por cada uno (61),
   cada una con su propia ubicación.
5. **Catálogo y ficheros, conectados.** Trino obtiene del metastore el
   esquema y las ubicaciones y lee los Parquet directamente de HDFS: los
   datos nunca pasan por el metastore.
6. **`DROP TABLE` sobre una tabla externa** borra sus filas del catálogo y
   deja intactos los ficheros. Lo comprobarás al final de la sesión con
   `drop_demo_external`; con una tabla administrada, en cambio, `DROP`
   borra también sus ficheros.

`DBS`, `TBLS`, `SDS`, `COLUMNS_V2`, `PARTITION_KEYS` y `PARTITIONS` son los
nombres reales de esas tablas en PostgreSQL: en la actividad «Mira dentro del
metastore» las consultarás con `psql`.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">¿Quién conserva los ficheros?</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Localiza los datos de una tabla externa y los de una administrada.</li><li style='margin:10px 0;list-style-position:outside'>Predice qué directorio seguirá en HDFS después de DROP TABLE.</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Tabla externa frente a tabla administrada

Se van a usar dos modelos distintos a lo largo de la sesión.

### Tabla externa sobre el data lake

Una tabla externa registra la ubicación de Parquet que **ya existe** en
`/datalake`. El catálogo añade nombre, esquema, formato y particiones, pero
la fuente de datos continúa siendo externa al ciclo de vida de la tabla: si
se elimina la definición de la tabla del metastore, los ficheros Parquet
siguen en HDFS exactamente igual que antes.

Este modelo es el adecuado para los datos fuente y refinados de S2 y S5:

```text
/datalake/raw/tpcds/item
    → tabla Hive externa: tcdm.item

/datalake/silver/tpcds/web_sales
    → tabla Hive externa particionada: tcdm.web_sales_silver

/datalake/gold/tpcds/web_sales_by_year
    → tabla Hive externa orientada al consumo: tcdm.web_sales_by_year_gold
```

Eliminar la definición de una de estas tablas **no** debe interpretarse como
permiso para borrar el data lake. Las operaciones destructivas sobre HDFS
deben ser explícitas y documentarse aparte, como se hace al final de este
notebook.

### Tabla administrada en el warehouse

Una tabla administrada deja que el catálogo y el motor elijan y gestionen
una ubicación bajo `/warehouse`. Al crearla, Hive/Trino reservan un
directorio con el patrón `/warehouse/<esquema>.db/<tabla>` — de ahí el
`/warehouse/tcdm.db/...` que verás más abajo. Al eliminar una tabla administrada,
el motor sí borra también esos ficheros: el ciclo de vida de los datos queda
ligado al ciclo de vida de la tabla.

En el laboratorio se usará esta zona para una única tabla de trabajo
(`tcdm.web_sales_by_year_managed_demo`), creada sólo para comparar ambos
modelos; en S7 será también la zona de las tablas Iceberg.

### Por qué interesa que el warehouse tenga un único dueño

En una tabla administrada el catálogo y el motor son los únicos dueños del
directorio: eligen la ubicación, escriben cada fichero, lo borran y nadie más
toca ahí. Eso da tres ventajas que una tabla externa, cuyo directorio
pertenece a otro proceso, no puede dar:

- **Un único ciclo de vida.** Crear la tabla y sus datos, o borrar la tabla y
  sus datos, es una sola operación: no quedan ficheros huérfanos ni nombres
  sin datos.
- **El catálogo sabe qué ficheros son de la tabla**, porque los ha escrito él.
  En una tabla externa los ha escrito otro (Spark en S5) y por eso hará falta
  más abajo un paso de descubrimiento de particiones.
- **Puede guardar sus propios metadatos junto a los datos.** Ésta es la
  ventaja que aprovecha Iceberg: una tabla Iceberg es una tabla administrada
  cuyo directorio contiene, además de los Parquet, metadatos propios que
  registran cada escritura como una versión nueva de la tabla. Como sólo el
  motor escribe ahí, puede hacer cada escritura **atómica** (un lector ve la
  tabla antes o después de escribir, nunca a medias) y **aislada** de otras
  escrituras. Eso, junto con la consistencia y la durabilidad, son las
  garantías **ACID** de una transacción de base de datos. Los datos siguen
  siendo Parquet en HDFS y otras herramientas pueden leerlos, pero qué
  ficheros forman la tabla en cada momento lo decide el catálogo con los
  metadatos de Iceberg, no el contenido del directorio. S7 lo muestra.

Ser administrada es una condición necesaria, no suficiente. La tabla
administrada de esta sesión es una tabla Hive: sus ficheros los gestiona el
motor, pero no hay versiones ni transacciones sobre ellos. Para eso hace
falta, además, el formato de tabla de S7.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #0c304d;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#0c304d" size="8" noshade style="display:none"><div style="color:#0c304d;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#0c304d" style="color:inherit"><b>Pregunta guía</b></font></div><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>¿Qué guarda el metastore y qué no? Si se perdiera PostgreSQL, ¿qué perderías: los datos de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>web_sales</code> o los nombres de las tablas?</li><li style='margin:10px 0;list-style-position:outside'>¿Quién decide dónde viven los ficheros de una tabla externa? ¿Y los de una administrada?</li></ul><hr color="#0c304d" size="2" noshade style="display:none"></div>

## Registrar los Parquet originales: `tcdm.item`

Empezamos por una tabla pequeña, `item`, exactamente el mismo Parquet
que S2 y S4 ya leyeron directamente por ruta
(`/datalake/raw/tpcds/item`, 36.000 filas).

`hive.tcdm` ya debería existir — lo crea `catalog-init` al levantar el
entorno—, pero repetir `CREATE SCHEMA IF NOT EXISTS` aquí es idempotente y
deja el paso explícito en el propio notebook.

El `CREATE TABLE` siguiente lista todas las columnas de `item` con su
tipo. El conector Hive de Trino no lee el esquema de los Parquet para crear
la tabla: hay que declararlo, y por eso lo copiamos del catálogo de S2 (el
`%esquema item` que precede al `CREATE TABLE`). Si lo declarado no
coincide con los ficheros, el error no aparece al crear la tabla sino al
leerla.

In [ ]:
run_trino("CREATE SCHEMA IF NOT EXISTS hive.tcdm")

In [ ]:
# Las columnas que declara el CREATE TABLE siguiente, con su significado.
%esquema item

In [ ]:
run_trino("DROP TABLE IF EXISTS hive.tcdm.item")

create_item_sql: str = '''
CREATE TABLE hive.tcdm.item (
    i_item_sk BIGINT,
    i_item_id VARCHAR,
    i_rec_start_date DATE,
    i_rec_end_date DATE,
    i_item_desc VARCHAR,
    i_current_price DECIMAL(7, 2),
    i_wholesale_cost DECIMAL(7, 2),
    i_brand_id BIGINT,
    i_brand VARCHAR,
    i_class_id BIGINT,
    i_class VARCHAR,
    i_category_id BIGINT,
    i_category VARCHAR,
    i_manufact_id BIGINT,
    i_manufact VARCHAR,
    i_size VARCHAR,
    i_formulation VARCHAR,
    i_color VARCHAR,
    i_units VARCHAR,
    i_container VARCHAR,
    i_manager_id BIGINT,
    i_product_name VARCHAR
) WITH (
    format = 'PARQUET',
    external_location = 'hdfs://namenode:9000/datalake/raw/tpcds/item'
)
'''
run_trino(create_item_sql)
print("tcdm.item creada.")

In [ ]:
run_trino("SHOW CREATE TABLE hive.tcdm.item")

In [ ]:
run_trino("DESCRIBE hive.tcdm.item")

In [ ]:
item_count: pd.DataFrame = run_trino("SELECT count(*) AS filas FROM hive.tcdm.item")
print(item_count)

assert int(item_count.loc[0, "filas"]) == 36_000

Un metastore no sólo guarda esquema y ubicación: también puede guardar un
comentario de tabla. Lo añadimos a `item` con `COMMENT ON TABLE` y lo
comprobamos en `SHOW CREATE TABLE`. Es un ejemplo de cómo documentar una
tabla; al final resumiremos por qué conviene dar propietario y descripción
a cada tabla del catálogo.

In [ ]:
run_trino('''
    COMMENT ON TABLE hive.tcdm.item IS
    'Dimensión de productos de TPC-DS SF3, registrada como tabla externa sobre /datalake/raw/tpcds/item (ver S2).'
''')
run_trino("SHOW CREATE TABLE hive.tcdm.item")

El `CREATE TABLE` no ha tardado más que declarar el esquema: no ha
leído ni copiado ningún byte de `item`. `SHOW CREATE TABLE` confirma que
`external_location` apunta exactamente a la misma ruta HDFS que ya conocías
de S2 y S4, y el recuento de filas coincide con el que obtuvo Spark en S4
leyendo el Parquet directamente. Registrar una tabla no transforma ni
duplica los datos: sólo añade un nombre y un esquema por encima de lo que ya
existía.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #a54467;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a54467" size="8" noshade style="display:none"><div style="color:#a54467;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a54467" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Registrar no es copiar</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>CREATE TABLE</code> no lee ni copia los datos de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>/datalake</code></li><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>SHOW CREATE TABLE</code> muestra la misma ruta que ya conocías de S2/S4</li><li style='margin:10px 0;list-style-position:outside'>El recuento de filas coincide, se lea por catálogo o por ruta</li></ul><hr color="#a54467" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Descubrir las particiones de silver</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Primero contaremos las ventas con la definición recién creada.</li><li style='margin:10px 0;list-style-position:outside'>Después compararemos el recuento con los meses registrados en el metastore.</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Registrar la versión particionada: `tcdm.web_sales_silver`

Ahora registramos la salida particionada de S5:
`/datalake/silver/tpcds/web_sales`, organizada en directorios
`sold_year=<año>/sold_month=<mes>/`. Las columnas, su orden y sus tipos
son los del esquema que S5 declaró como contrato de esta salida
(`web_sales_silver_schema`) y comprobó al escribirla y al releerla,
escritos con los nombres de tipo de Trino: el `timestamp` de Spark es
aquí un `TIMESTAMP` de Trino, que no lleva zona; los valores son los que S5
escribió, en UTC. Las columnas de negocio
son las del catálogo de `web_sales` que documentó S2, salvo
`ws_sold_date`, un `DATE` que S5 derivó de `ws_sold_timestamp`. Las dos últimas columnas, `sold_year` y `sold_month`, son las columnas de
partición que
introdujo S5 y **no** están dentro de los ficheros Parquet: Spark las quitó
de los datos al escribir con `partitionBy` y las dejó codificadas sólo en el
nombre de cada directorio. Por eso deben declararse en `partitioned_by` y,
en el listado de columnas de Trino, ir al final.

In [ ]:
%%esquema
# Las columnas de web_sales que conserva la tabla silver de S5.
ws_order_number
ws_item_sk
ws_bill_customer_sk
ws_bill_addr_sk
ws_sold_timestamp
ws_quantity
ws_list_price
ws_ext_discount_amt
ws_ext_sales_price
ws_net_paid
ws_net_profit

In [ ]:
run_trino("DROP TABLE IF EXISTS hive.tcdm.web_sales_silver")

create_web_sales_silver_sql: str = '''
CREATE TABLE hive.tcdm.web_sales_silver (
    ws_order_number BIGINT,
    ws_item_sk BIGINT,
    ws_bill_customer_sk BIGINT,
    ws_bill_addr_sk BIGINT,
    ws_sold_timestamp TIMESTAMP,
    ws_sold_date DATE,
    ws_quantity BIGINT,
    ws_list_price DECIMAL(7, 2),
    ws_ext_discount_amt DECIMAL(7, 2),
    ws_ext_sales_price DECIMAL(7, 2),
    ws_net_paid DECIMAL(7, 2),
    ws_net_profit DECIMAL(7, 2),
    sold_year INTEGER,
    sold_month INTEGER
) WITH (
    format = 'PARQUET',
    external_location = 'hdfs://namenode:9000/datalake/silver/tpcds/web_sales',
    partitioned_by = ARRAY['sold_year', 'sold_month']
)
'''
run_trino(create_web_sales_silver_sql)
print("tcdm.web_sales_silver creada.")

In [ ]:
run_trino("SHOW CREATE TABLE hive.tcdm.web_sales_silver")

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #0c304d;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#0c304d" size="8" noshade style="display:none"><div style="color:#0c304d;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#0c304d" style="color:inherit"><b>Pregunta guía</b></font></div><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Los Parquet de S5 llevan días en HDFS y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>count(*)</code> devuelve 0: ¿qué falta, los datos o los metadatos?</li><li style='margin:10px 0;list-style-position:outside'>¿Por qué <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>CREATE TABLE</code> no recorre HDFS por su cuenta para descubrir las particiones?</li></ul><hr color="#0c304d" size="2" noshade style="display:none"></div>

### Por qué hace falta un paso de descubrimiento de particiones

Consultamos la tabla recién creada antes de hacer nada más:

**Predicción para explicar en clase.** El directorio sigue en HDFS y la tabla acaba de registrarse. ¿Cuántas filas podrá encontrar Trino antes de registrar los meses? Después compara tu predicción con el primer recuento, el segundo y una fila de `$partitions`. Señala qué evidencia corresponde a los ficheros y cuál al catálogo.

In [ ]:
before_repair: pd.DataFrame = run_trino("SELECT count(*) AS filas FROM hive.tcdm.web_sales_silver")
print(before_repair)

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Hacer visibles los meses</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Los directorios ya existen en HDFS. Mira qué conoce el catálogo antes de sincronizar.</li><li style='margin:10px 0;list-style-position:outside'>Repite el recuento después de registrar las particiones.</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

El resultado es **0 filas**, aunque los ficheros Parquet de S5 llevan
ahí desde que terminó esa sesión. `CREATE TABLE ... WITH (external_location
= ..., partitioned_by = ...)` declara el esquema y la convención de
particionado, pero no recorre HDFS para descubrir qué valores de
`sold_year`/`sold_month` existen realmente: el metastore todavía no conoce
ninguna partición.

**Cómo se sincronizan las particiones desde Trino.** El patrón clásico
de Hive es `MSCK REPAIR TABLE`, pero esa sentencia es HiveQL, no SQL de
Trino: el [conector Hive de Trino](https://trino.io/docs/current/connector/hive.html)
no la implementa. Lo que expone en su lugar es un procedimiento propio del
conector, `system.sync_partition_metadata`, dentro del esquema `system` de
cada catálogo Hive — aquí, `hive.system.sync_partition_metadata`. Por eso
usamos esa llamada, y no `MSCK REPAIR TABLE`, para pedirle a Trino que
sincronice el metastore con los directorios que encuentre bajo
`external_location`. Acabamos de borrar y recrear esta definición, así que
el metastore no conserva particiones anteriores: basta con añadir las que
ya existen en HDFS. El parámetro `mode` indica qué sincronizar: `ADD`
registra las particiones que existen en HDFS y faltan en el metastore,
`DROP` elimina las del metastore cuyo directorio ya no existe, y `FULL`
hace las dos cosas.

In [ ]:
sync_partitions_sql: str = '''
CALL hive.system.sync_partition_metadata(
    schema_name => 'tcdm',
    table_name => 'web_sales_silver',
    mode => 'ADD')
'''
run_trino(sync_partitions_sql)
print("Particiones sincronizadas.")

In [ ]:
after_repair: pd.DataFrame = run_trino("SELECT count(*) AS filas FROM hive.tcdm.web_sales_silver")
print(after_repair)

assert int(after_repair.loc[0, "filas"]) > 0
assert int(before_repair.loc[0, "filas"]) == 0

Ahora sí aparecen filas: el metastore ha añadido una entrada de
partición por cada directorio `sold_year=.../sold_month=...` que ha
encontrado bajo la ubicación externa. Una partición Hive es exactamente esa
combinación: una organización física de directorios **más** una fila de
metadatos en el metastore que la describe. Sin la segunda parte, la primera
es invisible para el motor SQL, aunque los bytes ya estén en HDFS.

> **Nota para cuando uses Spark más adelante.** Spark, al ser también un
> cliente compatible con HiveQL, sí entiende `MSCK REPAIR TABLE` de forma
> nativa (`spark.sql("MSCK REPAIR TABLE tcdm.web_sales_silver")`). No hace
> falta ejecutarlo aquí porque Trino ya sincronizó las particiones en el
> mismo metastore que Spark va a leer; si lo lanzaras ahora sería una
> operación redundante que no encontraría nada nuevo que añadir.

### Inspeccionar las particiones con `$partitions`

Trino expone una tabla de metadatos especial, `<tabla>$partitions`, con una
fila por partición conocida y sus columnas de partición como columnas
normales.

In [ ]:
partitions_df: pd.DataFrame = run_trino(
    'SELECT * FROM hive.tcdm."web_sales_silver$partitions" ORDER BY sold_year, sold_month'
)
print(f"Particiones registradas: {len(partitions_df)}")
partitions_df

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #a54467;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a54467" size="8" noshade style="display:none"><div style="color:#a54467;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a54467" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Una partición son dos cosas</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Un directorio <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>columna=valor</code> en HDFS, más una fila en el metastore</li><li style='margin:10px 0;list-style-position:outside'>Trino no admite <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>MSCK REPAIR TABLE</code>: usa <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>hive.system.sync_partition_metadata</code></li><li style='margin:10px 0;list-style-position:outside'>Spark sí entiende <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>MSCK</code>, pero aquí el metastore ya está sincronizado</li></ul><hr color="#a54467" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #2f6f4f;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#2f6f4f" size="8" noshade style="display:none"><div style="color:#2f6f4f;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#2f6f4f" style="color:inherit"><b>Actividades</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Mira dentro del metastore</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Abre <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>psql</code> en el contenedor de PostgreSQL y lista <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>&quot;TBLS&quot;</code>: ¿qué <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>TBL_TYPE</code> tiene cada tabla de la sesión?</li><li style='margin:10px 0;list-style-position:outside'>Cuenta las filas de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>&quot;PARTITIONS&quot;</code> de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>web_sales_silver</code> y compáralas con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>&#36;partitions</code>: ¿coinciden?</li><li style='margin:10px 0;list-style-position:outside'>Busca en <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>&quot;SDS&quot;</code> el <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>LOCATION</code> de una tabla externa y el de una administrada</li><li style='margin:10px 0;list-style-position:outside'>¿Hay filas de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>web_sales</code> dentro de PostgreSQL? ¿Qué guarda entonces el metastore?</li></ul><hr color="#2f6f4f" size="2" noshade style="display:none"></div>

Para esta actividad, en una terminal de tu equipo (no en una celda de este
notebook), entra en la base de datos del metastore:

```bash
docker exec -it postgresql-metastore psql -U hive -d metastore_db
```

Los nombres de las tablas del metastore van en mayúsculas y hay que
entrecomillarlos, por ejemplo `SELECT "TBL_NAME", "TBL_TYPE" FROM "TBLS";`.
Para salir, `\q`.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Leer el plan</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Compara la consulta de todas las ventas con la de un solo mes.</li><li style='margin:10px 0;list-style-position:outside'>Busca el año y el mes en la lectura de la tabla.</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

### Consulta sin filtro frente a consulta filtrada por partición

Comparamos el plan de una consulta que recorre toda la tabla con el de otra
que restringe `sold_year` y `sold_month` — las columnas de partición, no una
transformación sobre ellas. `EXPLAIN` muestra el plan sin ejecutar la
consulta; `EXPLAIN ANALYZE` la ejecuta de verdad y añade estadísticas reales
de filas y bytes leídos por cada operador.

In [ ]:
explain_unfiltered: pd.DataFrame = run_trino(
    "EXPLAIN SELECT count(*) FROM hive.tcdm.web_sales_silver"
)
print("\n".join(explain_unfiltered.iloc[:, 0].tolist()))

In [ ]:
explain_filtered: pd.DataFrame = run_trino(
    '''EXPLAIN SELECT count(*) FROM hive.tcdm.web_sales_silver
    WHERE sold_year = 2000 AND sold_month = 1'''
)
print("\n".join(explain_filtered.iloc[:, 0].tolist()))

Busca en el segundo plan una restricción sobre `sold_year`/`sold_month`
en el propio `TableScan` (una condición de partición, no un `Filter`
posterior a la lectura): esa es la señal de que Trino ha descartado el resto
de directorios de partición antes de abrir ningún fichero, en lugar de leer
todo y filtrar después. `EXPLAIN ANALYZE` deja ver además cuántas filas y
bytes se leyeron realmente en cada caso.

In [ ]:
explain_analyze_filtered: pd.DataFrame = run_trino(
    '''EXPLAIN ANALYZE SELECT count(*) FROM hive.tcdm.web_sales_silver
    WHERE sold_year = 2000 AND sold_month = 1'''
)
print("\n".join(explain_analyze_filtered.iloc[:, 0].tolist()))

Igual que en S4 con *predicate/projection pushdown* en Spark, la
mejora de una consulta filtrada por partición no se debe dar por hecha sólo
por el tiempo de pared: hay que comprobarla en el plan y en las métricas de
filas y bytes leídos que ofrece `EXPLAIN ANALYZE`.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">La capa gold y las estadísticas</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Un agregado de negocio registrado como tabla, no recalculado aquí</li><li style='margin:10px 0;list-style-position:outside'>El optimizador no estima solo: hay que pedirle <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>ANALYZE</code></li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Registrar la tabla gold: `tcdm.web_sales_by_year_gold`

Por último registramos el agregado de negocio que produjo S5:
`/datalake/gold/tpcds/web_sales_by_year`, sin particiones — es una tabla
pequeña, un año por fila, pensada para consumirse entera. Sus columnas son
`sold_year`, `pedidos` (pedidos distintos), `lineas_de_venta` (líneas de venta,
`count(*)`) y `beneficio_neto`, con los tipos del esquema que S5 declaró
para `gold` (`web_sales_by_year_schema`) y comprobó al releer el fichero.
`beneficio_neto` es un `DECIMAL(17, 2)`: un `SUM` de una columna
`decimal(7,2)` en Spark amplía la precisión, pero mantiene el tipo decimal en
lugar de convertirlo a coma flotante — importante para un agregado de dinero.

In [ ]:
run_trino("DROP TABLE IF EXISTS hive.tcdm.web_sales_by_year_gold")

create_gold_sql: str = '''
CREATE TABLE hive.tcdm.web_sales_by_year_gold (
    sold_year INTEGER,
    pedidos BIGINT,
    lineas_de_venta BIGINT,
    beneficio_neto DECIMAL(17, 2)
) WITH (
    format = 'PARQUET',
    external_location = 'hdfs://namenode:9000/datalake/gold/tpcds/web_sales_by_year'
)
'''
run_trino(create_gold_sql)
print("tcdm.web_sales_by_year_gold creada.")

In [ ]:
run_trino("DESCRIBE hive.tcdm.web_sales_by_year_gold")

In [ ]:
gold_df: pd.DataFrame = run_trino(
    "SELECT * FROM hive.tcdm.web_sales_by_year_gold ORDER BY sold_year"
)
gold_df

`DESCRIBE` muestra el esquema declarado en el `CREATE TABLE`, no el de los
ficheros. Si el tipo declarado para `beneficio_neto` no coincidiera con el
del Parquet físico, la señal aparecería al leer: Trino se quejaría en el
`SELECT` anterior o los valores saldrían como `NULL`. El conector Hive de
Trino asocia columnas por nombre y espera un tipo compatible con el que
encuentra en el fichero; no infiere el esquema por sí solo.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Inspeccionar las tablas en Trino</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Lista los nombres registrados y consulta las estadísticas de silver.</li><li style='margin:10px 0;list-style-position:outside'>Observa qué estimaciones aparecen después de ANALYZE.</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Consultar con Trino

Con las tres tablas registradas, exploramos el catálogo desde Trino como lo
haría cualquier herramienta de BI o cualquier persona analista que sólo
conozca el nombre lógico de las tablas, sin saber nada de rutas HDFS.

In [ ]:
run_trino("SHOW SCHEMAS FROM hive")

In [ ]:
run_trino("SHOW TABLES FROM hive.tcdm")

### Estadísticas antes y después de `ANALYZE`

El metastore puede guardar estadísticas por columna (número de valores
distintos aproximado, fracción de nulos, tamaño medio...) que Trino usa al
estimar el coste de un plan. En consultas con `JOIN`, esas estimaciones
pueden influir en su orden; aquí veremos cómo cambian las estimaciones de
filas de una consulta filtrada. Las estadísticas no se calculan solas: hay
que pedirlas explícitamente con `ANALYZE`.

In [ ]:
run_trino("SHOW STATS FOR hive.tcdm.web_sales_silver")

In [ ]:
run_trino("ANALYZE hive.tcdm.web_sales_silver")

In [ ]:
run_trino("SHOW STATS FOR hive.tcdm.web_sales_silver")

In [ ]:
explain_filtered_tras_analyze: pd.DataFrame = run_trino(
    '''EXPLAIN SELECT count(*) FROM hive.tcdm.web_sales_silver
    WHERE sold_year = 2000 AND sold_month = 1'''
)
print("\n".join(explain_filtered_tras_analyze.iloc[:, 0].tolist()))

Compara ambas tablas de estadísticas. Antes de `ANALYZE`, varias columnas
aparecen vacías o con estimaciones toscas; después, Trino muestra
estimaciones por columna y `row_count` en la fila final. `SHOW STATS` no
sustituye a un recuento exacto como `COUNT(DISTINCT ...)`. En una tabla
Hive particionada, los valores distintos pueden diferir del total real;
la estimación global puede depender de cómo Trino agrega las estadísticas
de cada partición. No interpretes esa
diferencia como filas perdidas. Antes de `ANALYZE`, los `EXPLAIN` de más
arriba planificaron sin estas estadísticas de columna. El `EXPLAIN`
repetido deja ver en `Estimates` cómo el optimizador recibe una estimación
de filas en lugar de `?`. Esta consulta sólo tiene un recuento: no muestra
un cambio de orden de `JOIN` ni de agregación. Las estadísticas son una
foto del momento; si llegan particiones nuevas o cambian los datos, vuelve
a lanzar `ANALYZE`.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #a54467;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a54467" size="8" noshade style="display:none"><div style="color:#a54467;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a54467" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Las estadísticas no se calculan solas</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>SHOW STATS</code> antes de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>ANALYZE</code>: vacío o estimaciones toscas</li><li style='margin:10px 0;list-style-position:outside'>Después: estimaciones por columna y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>row_count</code>; los valores distintos pueden ser aproximados</li><li style='margin:10px 0;list-style-position:outside'>El <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>EXPLAIN</code> muestra las estimaciones disponibles para el planificador</li></ul><hr color="#a54467" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Spark como segundo cliente</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Ni una ruta HDFS en el código: sólo el nombre de la tabla</li><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>DESCRIBE FORMATTED</code> es el equivalente en Spark de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>SHOW CREATE TABLE</code></li><li style='margin:10px 0;list-style-position:outside'>La caché de particiones del driver y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>refreshTable</code></li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Consultar la misma tabla con Spark

Ahora usamos la `SparkSession` que configuramos al principio del notebook
para leer **la misma definición de tabla**, no para releer el Parquet por
ruta como en S4. La diferencia es la que separa "Spark lee ficheros" de
"Spark es un cliente del catálogo": aquí no aparece ninguna ruta HDFS en el
código, sólo el nombre lógico `tcdm.web_sales_silver`.

**Conecta las evidencias.** Identifica el directorio de S5, el mes registrado en `$partitions` y el filtro que aparecerá en el plan de Spark. ¿Por qué cambia el recuento después de sincronizar si la sincronización no escribe ventas?

In [ ]:
spark.sql("SHOW TABLES IN tcdm").show()

In [ ]:
from pyspark.sql import DataFrame

web_sales_silver_spark: DataFrame = spark.table("tcdm.web_sales_silver")
web_sales_silver_spark.printSchema()

In [ ]:
spark.sql('''SELECT sold_year, sold_month, count(*) AS lineas_de_venta
    FROM tcdm.web_sales_silver
    GROUP BY sold_year, sold_month
    ORDER BY sold_year, sold_month''').show(100)

### Comparar ubicación y esquema

`DESCRIBE FORMATTED` en Spark es el equivalente de `SHOW CREATE TABLE` en
Trino: expone la ubicación física, el formato de fichero y las columnas de
partición tal como las ve el mismo metastore.

In [ ]:
spark.sql("DESCRIBE FORMATTED tcdm.web_sales_silver").show(100, truncate=False)

Busca en la salida la línea `Location`: debe ser la misma ruta HDFS
que declaraste como `external_location` en el `CREATE TABLE` de Trino, y las
líneas `Partition Provider` / `# Partition Information` deben mencionar
`sold_year` y `sold_month`. Spark no ha vuelto a preguntar "dónde están los
datos": lo ha leído del mismo metastore que consultó Trino.

### Comparar el resultado de negocio

Repetimos, con Spark, la misma pregunta de negocio que ya respondió Trino
sobre `tcdm.web_sales_by_year_gold`. El cruce externo conserva los años
que aparezcan en un solo resultado, y la comprobación de unicidad asegura
que comparamos una fila por año en cada motor.

In [ ]:
gold_spark_df: pd.DataFrame = spark.sql(
    "SELECT sold_year, pedidos, lineas_de_venta, beneficio_neto FROM tcdm.web_sales_by_year_gold ORDER BY sold_year"
).toPandas()
gold_spark_df

In [ ]:
assert not gold_df.empty and not gold_spark_df.empty

comparacion: pd.DataFrame = gold_df.merge(
    gold_spark_df,
    on="sold_year",
    how="outer",
    suffixes=("_trino", "_spark"),
    validate="one_to_one",
    indicator=True,
)
assert comparacion["_merge"].eq("both").all(), (
    "Los años de Trino y Spark no coinciden: revisa la columna _merge."
)
assert len(comparacion) == len(gold_df) == len(gold_spark_df)
comparacion = comparacion.drop(columns="_merge")
comparacion["beneficio_neto_trino"] = comparacion["beneficio_neto_trino"].astype(float)
comparacion["beneficio_neto_spark"] = comparacion["beneficio_neto_spark"].astype(float)
comparacion["diferencia_beneficio"] = (
    comparacion["beneficio_neto_trino"] - comparacion["beneficio_neto_spark"]
).abs()

assert (comparacion["pedidos_trino"] == comparacion["pedidos_spark"]).all()
assert (comparacion["lineas_de_venta_trino"] == comparacion["lineas_de_venta_spark"]).all()
assert (comparacion["diferencia_beneficio"] < 0.01).all()
print("Trino y Spark devuelven el mismo resultado de negocio sobre tcdm.web_sales_by_year_gold.")
comparacion

### Plan de lectura y caché de metadatos

`explain()` en Spark, igual que `EXPLAIN` en Trino, deja ver si el filtro por
columnas de partición se resuelve podando particiones antes de leer ningún
fichero (`PartitionFilters` en el plan) o si Spark tiene que abrir todos los
ficheros y filtrar después.

In [ ]:
spark.sql("SELECT * FROM tcdm.web_sales_silver WHERE sold_year = 2000 AND sold_month = 1").explain(
    "formatted"
)

Busca `PartitionFilters` en la sección del `Scan`: debe incluir la
condición sobre `sold_year`/`sold_month`, la misma poda que ya viste en el
plan de Trino.

Un último aviso sobre las cachés de metadatos: Spark guarda en memoria del driver la lista de particiones que ya
ha visto para una tabla externa. Si otro cliente —Trino, Hive o el propio
`sync_partition_metadata`— añade particiones nuevas después de que Spark ya
haya consultado la tabla en esta sesión, Spark no las verá hasta que se le
pida explícitamente que refresque esa tabla:

```python
spark.catalog.refreshTable("tcdm.web_sales_silver")
```

No hace falta ejecutarlo ahora —no hemos añadido particiones nuevas después
de leer la tabla con Spark—, pero es el motivo por el que a veces una tabla
"no aparece actualizada" en Spark aunque Trino ya vea los datos nuevos.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #0c304d;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#0c304d" size="8" noshade style="display:none"><div style="color:#0c304d;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#0c304d" style="color:inherit"><b>Pregunta guía</b></font></div><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Spark y Trino comparten metastore. Si Trino registra una partición nueva mientras Spark tiene la tabla abierta, ¿la verá Spark?</li><li style='margin:10px 0;list-style-position:outside'>¿Cómo lo comprobarías y qué orden darías para que la viera?</li></ul><hr color="#0c304d" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Seguir una copia administrada</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Crearemos una tabla con las seis filas anuales de gold.</li><li style='margin:10px 0;list-style-position:outside'>Busca el fichero nuevo en el warehouse y contrasta su ciclo de vida con el de la fuente.</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Tabla administrada: `tcdm.web_sales_by_year_managed_demo`

Para cerrar la comparación externa/administrada, creamos una tabla de
laboratorio con `CREATE TABLE ... AS SELECT` (CTAS) a partir de la tabla gold
que ya registramos. No se declara ningún `external_location`: dejamos que
Hive/Trino elijan la ubicación bajo `/warehouse`. La celda siguiente borra
primero la tabla administrada si ya existía; al repetirla también se borran
sus ficheros anteriores y se crea una copia nueva.

In [ ]:
run_trino("DROP TABLE IF EXISTS hive.tcdm.web_sales_by_year_managed_demo")

ctas_sql: str = '''
CREATE TABLE hive.tcdm.web_sales_by_year_managed_demo
WITH (format = 'PARQUET') AS
SELECT * FROM hive.tcdm.web_sales_by_year_gold
'''
run_trino(ctas_sql)
print("tcdm.web_sales_by_year_managed_demo creada (tabla administrada).")

In [ ]:
run_trino("SHOW CREATE TABLE hive.tcdm.web_sales_by_year_managed_demo")

Compara este `SHOW CREATE TABLE` con el de `tcdm.web_sales_by_year_gold`:
la tabla administrada no declara `external_location`, y su ubicación real
—visible también en la propiedad `location` que añade Trino— cae bajo
`/warehouse/tcdm.db/web_sales_by_year_managed_demo`, siguiendo el patrón
`/warehouse/<esquema>.db/<tabla>` que se explicó en «Tabla administrada en
el warehouse».
Lo comprobamos directamente en HDFS:

In [ ]:
!hdfs dfs -ls -R -h /warehouse/tcdm.db/web_sales_by_year_managed_demo

Ahí sí hay ficheros Parquet nuevos, escritos por el propio `CTAS`: a
diferencia de las tablas externas de las secciones anteriores, esta tabla
**sí** copió y materializó los datos. Es la diferencia observable entre
registrar una ubicación existente y pedirle al motor que gestione una nueva.

### Qué ocurre al borrar cada tipo de tabla

La diferencia entre los dos modelos se ve al borrar. Para no tocar ninguna
tabla de la sesión, creamos dos tablas efímeras con los mismos datos de
`gold`: una administrada, cuya ubicación elige el catálogo, y una externa,
sobre una ruta de trabajo propia bajo `/user/luser/s6`.

In [ ]:
!hdfs dfs -mkdir -p /user/luser/s6
!hdfs dfs -rm -r -f /user/luser/s6/drop_demo_external

run_trino("DROP TABLE IF EXISTS hive.tcdm.drop_demo_managed")
run_trino("DROP TABLE IF EXISTS hive.tcdm.drop_demo_external")

run_trino('''
    CREATE TABLE hive.tcdm.drop_demo_managed
    WITH (format = 'PARQUET') AS
    SELECT * FROM hive.tcdm.web_sales_by_year_gold
''')
run_trino('''
    CREATE TABLE hive.tcdm.drop_demo_external
    WITH (
        format = 'PARQUET',
        external_location = 'hdfs://namenode:9000/user/luser/s6/drop_demo_external'
    ) AS
    SELECT * FROM hive.tcdm.web_sales_by_year_gold
''')
print("Tablas efímeras creadas: drop_demo_managed y drop_demo_external.")

Antes de borrar, las dos tienen ficheros en HDFS: la administrada bajo
`/warehouse/tcdm.db`, la externa en la ruta que le indicamos.

In [ ]:
!hdfs dfs -ls /warehouse/tcdm.db/drop_demo_managed /user/luser/s6/drop_demo_external

Ahora ejecutamos el mismo `DROP TABLE` sobre las dos y volvemos a mirar
HDFS.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #0c304d;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#0c304d" size="8" noshade style="display:none"><div style="color:#0c304d;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#0c304d" style="color:inherit"><b>Pregunta guía</b></font></div><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Vas a ejecutar el mismo <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>DROP TABLE</code> sobre una tabla administrada y sobre una externa: ¿qué esperas encontrar en HDFS tras cada una?</li><li style='margin:10px 0;list-style-position:outside'>¿Qué orden usarías para comprobarlo?</li></ul><hr color="#0c304d" size="2" noshade style="display:none"></div>

In [ ]:
import subprocess

run_trino("DROP TABLE hive.tcdm.drop_demo_managed")
run_trino("DROP TABLE hive.tcdm.drop_demo_external")


def hdfs_dir_exists(path: str) -> bool:
    return subprocess.run(["hdfs", "dfs", "-test", "-d", path]).returncode == 0


managed_dir_exists: bool = hdfs_dir_exists("/warehouse/tcdm.db/drop_demo_managed")
external_dir_exists: bool = hdfs_dir_exists("/user/luser/s6/drop_demo_external")
print(f"¿Sigue el directorio de la tabla administrada? {managed_dir_exists}")
print(f"¿Sigue el directorio de la tabla externa?      {external_dir_exists}")

assert not managed_dir_exists
assert external_dir_exists

In [ ]:
!hdfs dfs -ls /user/luser/s6/drop_demo_external

Las dos tablas han desaparecido del catálogo, pero sólo la administrada se
ha llevado sus ficheros: los de la externa siguen en HDFS, sin ninguna
definición que los describa. Por eso el `DROP TABLE` de una tabla externa no
es una forma de borrar datos, y por eso las definiciones de `/datalake` se
pueden eliminar y volver a crear sin riesgo. Borrar esos ficheros es una
decisión aparte:

In [ ]:
!hdfs dfs -rm -r /user/luser/s6/drop_demo_external

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #a54467;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a54467" size="8" noshade style="display:none"><div style="color:#a54467;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a54467" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Dos ciclos de vida distintos</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Tabla externa: <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>DROP TABLE</code> sólo borra el nombre del catálogo</li><li style='margin:10px 0;list-style-position:outside'>Tabla administrada: <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>DROP TABLE</code> borra también los Parquet bajo <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>/warehouse</code></li><li style='margin:10px 0;list-style-position:outside'>La comparación observable es <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>hdfs dfs -ls</code>, no el <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>SHOW CREATE TABLE</code></li><li style='margin:10px 0;list-style-position:outside'>Un único dueño de los ficheros es lo que permite a Iceberg dar ACID en S7</li></ul><hr color="#a54467" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #2f6f4f;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#2f6f4f" size="8" noshade style="display:none"><div style="color:#2f6f4f;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#2f6f4f" style="color:inherit"><b>Actividades</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Cambia las tablas y observa</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Crea tu propia tabla administrada, con Trino o con Spark, y averigua dónde guarda sus ficheros</li><li style='margin:10px 0;list-style-position:outside'>Copia dos meses de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver</code> a <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>/user/luser/s6/silver_copia</code>, regístralos como tabla externa particionada y sincroniza: ¿cuántas particiones ve?</li><li style='margin:10px 0;list-style-position:outside'>Añade un tercer mes a la copia y cuenta antes y después de sincronizar</li><li style='margin:10px 0;list-style-position:outside'>Declara <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>beneficio_neto</code> como <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>DOUBLE</code> en una tabla nueva sobre la ruta de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>gold</code>: ¿cuándo y cómo se queja Trino?</li><li style='margin:10px 0;list-style-position:outside'>Calcula <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>ANALYZE</code> de una sola partición (<code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>partitions = ARRAY[ARRAY[&#x27;2000&#x27;,&#x27;1&#x27;]]</code>) y mira <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>SHOW STATS</code>: ¿qué cambia?</li><li style='margin:10px 0;list-style-position:outside'>Borra las tablas nuevas al terminar; no toques las tres de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>/datalake</code></li></ul><hr color="#2f6f4f" size="2" noshade style="display:none"></div>

### Ejercicio: tu propia tabla administrada

Crea una tabla administrada de laboratorio, `tcdm.web_sales_lab_2000`, con las
ventas de `tcdm.web_sales_silver` del año 2000 (`sold_year = 2000`). Puedes
hacerlo con **Trino** o con **Spark**, a tu elección, pero sin indicar
ubicación: que sea el motor quien la elija.

- En Trino, un `CREATE TABLE ... WITH (format = 'PARQUET') AS SELECT ...`,
  como el de la tabla administrada anterior, y sin `external_location`.
- En Spark, un `CREATE TABLE ... STORED AS PARQUET AS SELECT ...` con
  `spark.sql(...)`, sin `LOCATION`.

Después, con el **otro** motor, comprueba lo siguiente:

1. Que ve la tabla y que su recuento de filas coincide con el de `silver`
   filtrada por `sold_year = 2000`.
2. Su ubicación (`SHOW CREATE TABLE` en Trino, `DESCRIBE FORMATTED` en
   Spark): ¿cuelga de `/warehouse/tcdm.db`?
3. Los ficheros que hay en esa ubicación, con `!hdfs dfs -ls -R`: ¿quién los
   ha escrito, tú o el motor? ¿Cuántos hay, en comparación con los de
   `silver` del año 2000?

Para terminar, borra la tabla con `DROP TABLE` y comprueba con `hdfs dfs -ls`
que su directorio ha desaparecido. No toques las tres tablas externas de
`/datalake`.


In [ ]:
# Tu solución: crea la tabla administrada, comprueba su ubicación y sus ficheros con el otro motor, y bórrala

**Tus observaciones:**

*Responde también: ¿qué ventaja te ha dado que sea el motor el único que
escribe en ese directorio? ¿Qué garantía no tendrías si dos procesos
escribieran en él a la vez?*


<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Linaje: de dónde sale cada tabla</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>El metastore sabe dónde está cada tabla, pero no de qué se calculó</li><li style='margin:10px 0;list-style-position:outside'>Lo apuntamos en sus propiedades, con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>TBLPROPERTIES</code></li><li style='margin:10px 0;list-style-position:outside'>Con eso, la lista de qué recalcular cuando cambia una fuente</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Linaje: de qué depende cada tabla

Las tablas de esta sesión no son independientes. S5 calculó `silver` a
partir de los Parquet de `raw` y `gold` a partir de `silver`, y la tabla
administrada de laboratorio salió de `gold` con un CTAS:

```text
/datalake/raw/tpcds/web_sales
    → tcdm.web_sales_silver                    (S5: filtra y añade sold_year y sold_month)
        → tcdm.web_sales_by_year_gold          (S5: agrega por año)
            → tcdm.web_sales_by_year_managed_demo   (S6: CTAS)
```

Esa cadena es el **linaje** (*lineage*) de cada tabla: de qué datos sale y
qué proceso la escribe. Sirve para responder una pregunta que llegará en S8,
cuando empiecen a llegar ventas nuevas a `raw`: **si cambia una fuente, ¿qué
tablas hay que recalcular, y en qué orden?** Y también la contraria: si un
número de `gold` está mal, ¿de qué datos viene?

El metastore guarda la ubicación de cada tabla, pero no quién escribió esos
ficheros ni a partir de qué: para él, `gold` es una tabla externa más. El
linaje hay que apuntarlo, y el catálogo es el sitio natural, porque es donde
todos los motores buscan las tablas. Cada tabla admite **propiedades**:
pares clave-valor que el metastore guarda con ella en su tabla
`TABLE_PARAMS` de PostgreSQL, la misma donde queda el comentario de
`COMMENT ON TABLE`. Usaremos dos claves propias del curso:

- `tcdm.linaje.origenes`: de qué tablas (`tcdm.…`) o rutas de HDFS se
  calcula, separadas por comas;
- `tcdm.linaje.proceso`: qué proceso la escribe, para saber qué volver a
  ejecutar.

Las fijamos desde Spark con `ALTER TABLE … SET TBLPROPERTIES`. `tcdm.item` no
lleva ninguna: no se calcula a partir de nada, sólo apunta a los Parquet de
`raw`.

**Antes de leer el código.** Dibuja raw → silver → gold e indica qué operación produjo cada flecha. Si cambian las ventas de raw, predice el orden de recalculo. Después compara tu respuesta con `afectadas`. La implementación del recorrido del grafo se puede estudiar en una segunda lectura.

In [ ]:
linaje: dict[str, dict[str, str]] = {
    "tcdm.web_sales_silver": {
        "tcdm.linaje.origenes": "hdfs://namenode:9000/datalake/raw/tpcds/web_sales",
        "tcdm.linaje.proceso": "S5: filtrar las ventas sin fecha y añadir sold_year y sold_month",
    },
    "tcdm.web_sales_by_year_gold": {
        "tcdm.linaje.origenes": "tcdm.web_sales_silver",
        "tcdm.linaje.proceso": "S5: agregar silver por año",
    },
    "tcdm.web_sales_by_year_managed_demo": {
        "tcdm.linaje.origenes": "tcdm.web_sales_by_year_gold",
        "tcdm.linaje.proceso": "S6: CTAS de la tabla gold",
    },
}

for tabla, propiedades in linaje.items():
    asignaciones: str = ", ".join(f"'{clave}' = '{valor}'" for clave, valor in propiedades.items())
    spark.sql(f"ALTER TABLE {tabla} SET TBLPROPERTIES ({asignaciones})")
    print(f"{tabla}: linaje apuntado en el metastore.")

Spark ha escrito las propiedades en el metastore, y Trino las lee de ahí. Su
conector Hive las expone en la tabla oculta `"<tabla>$properties"`, con una
columna por clave:

In [ ]:
run_trino('''
    SELECT "tcdm.linaje.origenes", "tcdm.linaje.proceso"
    FROM hive.tcdm."web_sales_by_year_gold$properties"
''')

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Predecir el orden de recálculo</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Parte de una venta modificada en raw y sigue sus dependencias.</li><li style='margin:10px 0;list-style-position:outside'>Contrasta tu predicción con la lista de tablas afectadas.</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

### Qué hay que recalcular si cambia una fuente

Con el linaje en el catálogo, la lista de dependencias se calcula en lugar de
recordarla. La celda siguiente recorre las tablas de `tcdm` y lee, de cada
una, su ubicación, si está particionada y sus dos propiedades de linaje. Una
tabla depende de un cambio de dos maneras:

- **lee directamente los ficheros que cambian**, porque su ubicación es esa
  ruta, como `tcdm.item` sobre `/datalake/raw/tpcds/item`. No hay que
  recalcularla: una tabla externa no copia los datos, sólo apunta a ellos. Si
  está particionada, sí hay que registrar las particiones nuevas, como
  hiciste con `sync_partition_metadata`;
- **se calcula a partir de lo que cambia**, porque lo nombra en
  `tcdm.linaje.origenes`. Hay que volver a ejecutar su proceso, y como sus
  ficheros cambian, lo mismo vale para las tablas que se calculan a partir
  de ella.

El orden importa: `gold` no se puede recalcular antes que `silver`. La
función ordena las tablas afectadas con `graphlib.TopologicalSorter`, de la
biblioteca estándar de Python, de modo que cada una aparece después de todas
las tablas de las que se calcula.

In [ ]:
from dataclasses import dataclass
from graphlib import TopologicalSorter
from urllib.parse import urlparse

from pyspark.errors import AnalysisException
from pyspark.sql import Row

CLAVE_ORIGENES: str = "tcdm.linaje.origenes"
CLAVE_PROCESO: str = "tcdm.linaje.proceso"


def nodo(referencia: str) -> str:
    '''Una tabla se nombra `tcdm.tabla`; una ruta, sin `hdfs://servidor:puerto`.'''
    referencia = referencia.strip()
    if "/" in referencia:
        return urlparse(referencia).path.rstrip("/")
    return referencia


@dataclass(frozen=True)
class TablaConLinaje:
    nombre: str
    ubicacion: str
    particionada: bool
    origenes: tuple[str, ...]
    proceso: str


def leer_linaje(base_de_datos: str) -> dict[str, TablaConLinaje]:
    '''Lee del metastore la ubicación y las propiedades de linaje de cada tabla.'''
    tablas: dict[str, TablaConLinaje] = {}
    for tabla in spark.catalog.listTables(base_de_datos):
        if tabla.isTemporary:
            continue
        nombre: str = f"{base_de_datos}.{tabla.name}"
        try:
            detalle: list[Row] = spark.sql(f"DESCRIBE TABLE EXTENDED {nombre}").collect()
            filas: list[Row] = spark.sql(f"SHOW TBLPROPERTIES {nombre}").collect()
        except AnalysisException as error:
            print(f"{nombre}: no se puede leer desde Spark ({error.getCondition()}), se omite.")
            continue
        propiedades: dict[str, str] = {fila["key"]: fila["value"] for fila in filas}
        ubicacion: str = next((f["data_type"] for f in detalle if f["col_name"] == "Location"), "")
        tablas[nombre] = TablaConLinaje(
            nombre=nombre,
            ubicacion=nodo(ubicacion),
            particionada=any(f["col_name"] == "# Partition Information" for f in detalle),
            origenes=tuple(
                nodo(origen)
                for origen in propiedades.get(CLAVE_ORIGENES, "").split(",")
                if origen.strip()
            ),
            proceso=propiedades.get(CLAVE_PROCESO, ""),
        )
    return tablas


def afectadas(cambio: str, tablas: dict[str, TablaConLinaje]) -> pd.DataFrame:
    '''Las tablas a las que llega un cambio en `cambio` (una tabla o una ruta), en orden.'''
    motivo: dict[str, str] = {}
    pendientes: list[str] = [nodo(cambio)]
    while pendientes:
        actual: str = pendientes.pop()
        for tabla in tablas.values():
            if tabla.nombre in motivo:
                continue
            if tabla.ubicacion == actual:
                motivo[tabla.nombre] = f"lee los ficheros de {actual}"
            elif actual in tabla.origenes:
                motivo[tabla.nombre] = f"se calcula a partir de {actual}"
            else:
                continue
            pendientes.append(tabla.nombre)
    # Cada tabla, después de todas las afectadas de las que se calcula.
    grafo: dict[str, list[str]] = {
        nombre: [origen for origen in tablas[nombre].origenes if origen in motivo]
        for nombre in motivo
    }
    filas: list[dict[str, str]] = []
    for nombre in TopologicalSorter(grafo).static_order():
        tabla: TablaConLinaje = tablas[nombre]
        if motivo[nombre].startswith("lee"):
            hacer: str = (
                "registrar las particiones nuevas"
                if tabla.particionada
                else "nada: ya ve los datos nuevos"
            )
        else:
            hacer = f"recalcular ({tabla.proceso or 'proceso sin apuntar'})"
            if tabla.particionada:
                hacer += " y registrar sus particiones"
        filas.append(
            {"tabla": nombre, "por qué le afecta": motivo[nombre], "qué hay que hacer": hacer}
        )
    return pd.DataFrame(filas)


tablas_linaje: dict[str, TablaConLinaje] = leer_linaje("tcdm")
pd.DataFrame(
    [
        {
            "tabla": t.nombre,
            "ubicación": t.ubicacion,
            "particionada": t.particionada,
            "orígenes": ", ".join(t.origenes),
        }
        for t in tablas_linaje.values()
    ]
)

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #0c304d;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#0c304d" size="8" noshade style="display:none"><div style="color:#0c304d;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#0c304d" style="color:inherit"><b>Pregunta guía</b></font></div><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Antes de ejecutar la celda siguiente: si llegan ventas nuevas a <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>/datalake/raw/tpcds/web_sales</code>, ¿qué tablas hay que recalcular, y en qué orden?</li><li style='margin:10px 0;list-style-position:outside'>¿Y si cambian los Parquet de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>item</code>?</li></ul><hr color="#0c304d" size="2" noshade style="display:none"></div>

In [ ]:
afectadas("hdfs://namenode:9000/datalake/raw/tpcds/web_sales", tablas_linaje)

Ventas nuevas en `raw` obligan a recalcular `silver` y registrar sus
particiones nuevas, después `gold` y por último la tabla de laboratorio, en
ese orden, aunque ni `gold` ni la tabla de laboratorio lean `raw`
directamente: el cambio les llega a través de las tablas de las que se
calculan. Compáralo con un cambio en los Parquet de `item`:

In [ ]:
afectadas("hdfs://namenode:9000/datalake/raw/tpcds/item", tablas_linaje)

`tcdm.item` ve los datos nuevos en cuanto existen, sin recalcular nada, y
ninguna tabla del curso se calcula a partir de ella, así que la lista
termina ahí.

En S8 los datos nuevos llegan a otra ruta de `raw`,
`/datalake/raw/streaming/web_sales_incremental`, y la sesión hace lo que dice
esta lista, pero de forma incremental y continua: cada micro-lote se fusiona
con `MERGE INTO` en la tabla *silver* de Iceberg y desde ella en la *gold*,
en ese orden, sin recalcularlas enteras.

Este linaje es **declarado**: lo hemos escrito a mano, y nada impide que se
quede viejo si alguien cambia el proceso de S5 sin actualizar las
propiedades. En una plataforma real lo **capturan los motores** al ejecutar
cada trabajo: Spark, Trino o Airflow informan de qué leyeron y qué
escribieron mediante OpenLineage, y catálogos como DataHub, OpenMetadata o
Unity Catalog dibujan el grafo de dependencias, incluso columna a columna. La
idea es la misma que aquí: el linaje vive en el catálogo, junto a las
tablas.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #a54467;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a54467" size="8" noshade style="display:none"><div style="color:#a54467;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a54467" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">El linaje, en el catálogo</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Propiedades de tabla: pares clave-valor en <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>TABLE_PARAMS</code>, visibles desde Spark y Trino</li><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>tcdm.linaje.origenes</code> y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>tcdm.linaje.proceso</code>: de qué sale cada tabla y quién la escribe</li><li style='margin:10px 0;list-style-position:outside'>Si cambia <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>raw</code>: recalcular <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver</code>, después <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>gold</code>, después lo que salga de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>gold</code></li><li style='margin:10px 0;list-style-position:outside'>Una tabla que sólo apunta a los ficheros no se recalcula</li></ul><hr color="#a54467" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #2f6f4f;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#2f6f4f" size="8" noshade style="display:none"><div style="color:#2f6f4f;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#2f6f4f" style="color:inherit"><b>Actividades</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Amplía el linaje</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Crea con un CTAS una tabla de laboratorio a partir de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver</code> (ventas por mes), apunta su linaje y busca dónde aparece en la lista al cambiar <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>raw</code></li><li style='margin:10px 0;list-style-position:outside'>En <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>psql</code>, consulta las filas de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>&quot;TABLE_PARAMS&quot;</code> de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>web_sales_by_year_gold</code>: ¿qué otras propiedades guarda el metastore?</li><li style='margin:10px 0;list-style-position:outside'>¿Qué devuelve <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>afectadas</code> para <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>tcdm.web_sales_silver</code>? ¿Y para la ruta de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>gold</code>?</li><li style='margin:10px 0;list-style-position:outside'>Borra la tabla de laboratorio al terminar</li></ul><hr color="#2f6f4f" size="2" noshade style="display:none"></div>

### Ejercicio: una tabla derivada con su linaje

Crea una tabla administrada de laboratorio, `tcdm.web_sales_lab_por_mes`, con
las ventas de `tcdm.web_sales_silver` agregadas por mes: `sold_year`,
`sold_month`, el número de líneas de venta y la suma de `ws_net_paid`. Hazlo
con **Trino** o con **Spark**, a tu elección.

Después, con Spark, que es quien sabe fijar propiedades en esta sesión:

1. Apunta su linaje con `ALTER TABLE ... SET TBLPROPERTIES`: de qué tabla
   se calcula (`tcdm.linaje.origenes`) y con qué proceso
   (`tcdm.linaje.proceso`).
2. Vuelve a leer el linaje del esquema con `leer_linaje("tcdm")` y pregunta a
   `afectadas` qué habría que recalcular si cambiara la ruta de `raw` de
   `web_sales`: ¿aparece tu tabla, y después de cuáles?
3. Comprueba desde Trino que sus propiedades se leen en
   `"web_sales_lab_por_mes$properties"`.

Para terminar, borra la tabla de laboratorio y vuelve a preguntar a `afectadas`:
¿qué cambia en la lista?


In [ ]:
# Tu solución: crea la tabla por mes, apunta su linaje, consulta afectadas y bórrala

**Tus observaciones:**

*Si no hubieras apuntado el linaje, ¿qué habría devuelto `afectadas` para tu
tabla? ¿Y si hubieras apuntado un origen equivocado?*


<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Registrar el resto del data lake</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Hasta ahora, tres tablas: un catálogo sólo vale lo que cubre</li><li style='margin:10px 0;list-style-position:outside'>Las 17 tablas de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>raw</code> que faltan, con el esquema que infiere Spark</li><li style='margin:10px 0;list-style-position:outside'>El tipo de fecha con hora que entienden los dos motores, y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver</code></li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Registrar todo `raw` con Spark

Un catálogo sólo ayuda a encontrar lo que tiene registrado. Con tres tablas,
`SHOW TABLES` apenas mejora un `hdfs dfs -ls`; con todas las del data lake,
el catálogo pasa a ser el sitio donde buscar. Registramos ahora las 17 tablas
de `/datalake/raw/tpcds` que faltan (`item` ya la tienes).

Hay que decidir dos cosas:

- **Quién las registra: Spark.** Con Trino habría que escribir las columnas de
  cada tabla, como hiciste con `item`. Un Parquet ya guarda sus columnas y
  sus tipos, así que dejamos que Spark los lea de los ficheros
  (`spark.read.parquet(ruta).schema`) y no tecleamos 17 esquemas.
- **Qué nombre: `raw_<tabla>`.** S7 creará en este mismo espacio de nombres
  `tcdm` tablas Iceberg como `web_sales`, y una tabla Hive con ese nombre
  chocaría con ella. El prefijo también deja ver en qué capa está cada tabla.

Aprovechamos para documentarlas: al esquema inferido le añadimos como
comentario la descripción del apéndice de S2, la misma que muestra
`%esquema`, tanto de la tabla como de cada columna. Eso permitirá buscar por
significado, no sólo por nombre.

Una precisión sobre los tipos, que conviene ver antes de registrar nada: DuckDB
guarda las fechas con hora como «timestamp sin zona», y Spark 4 las infiere
por defecto como `timestamp_ntz`. Probémoslo con `web_returns`, que tiene una.

In [ ]:
spark.sql("DROP TABLE IF EXISTS tcdm.raw_ntz_demo")
spark.sql(
    "CREATE TABLE tcdm.raw_ntz_demo USING parquet "
    "LOCATION 'hdfs://namenode:9000/datalake/raw/tpcds/web_returns'"
)
spark.sql("DESCRIBE tcdm.raw_ntz_demo").filter("col_name = 'wr_returned_timestamp'").show()
descripcion_trino: pd.DataFrame = run_trino("DESCRIBE hive.tcdm.raw_ntz_demo")
spark.sql("DROP TABLE IF EXISTS tcdm.raw_ntz_demo")
descripcion_trino

Spark ve `wr_returned_timestamp` como `timestamp_ntz`, y Trino no ve ninguna de
las columnas de la tabla: sólo una de relleno, `col`. Cuando una tabla lleva un
tipo que Hive no conoce, Spark la guarda en el metastore en un formato propio
que el metastore no sabe describir, y Trino, que lee el metastore, se queda sin
esquema.

La solución es no dejar que Spark infiera ese tipo: se lee el esquema de los
ficheros y se **fuerza** a `timestamp` todo campo `timestamp_ntz` antes de
crear la tabla. Es lo que hace la función `registrar` de la celda siguiente.
Los valores que se leen son los mismos mientras la zona horaria sea UTC, como
aquí. Así el registro tampoco depende de quién escribió el Parquet ni de la
versión de Spark que lo escribió.

In [ ]:
import subprocess

from pyspark.sql.types import StructField, StructType, TimestampNTZType, TimestampType
from tcdm_esquema import Schema, load_schema

HDFS_RAIZ: str = "hdfs://namenode:9000"
RAW_TPCDS: str = "/datalake/raw/tpcds"

apendice: Schema = load_schema()


def listar_directorios(patron: str) -> list[str]:
    '''Los directorios de HDFS que encajan con un patrón, como rutas sin servidor.'''
    salida: str = subprocess.run(
        ["hdfs", "dfs", "-ls", "-d", patron], capture_output=True, text=True, check=True
    ).stdout
    return sorted(linea.split()[-1] for linea in salida.splitlines() if linea.startswith("d"))


def literal(texto: str) -> str:
    '''Un texto como literal entre comillas simples de Spark SQL.'''
    return texto.replace("\\", "\\\\").replace("'", "\\'")


def registrar(
    tabla: str,
    ruta: str,
    descripcion: str = "",
    comentarios: dict[str, str] | None = None,
    particiones: tuple[str, ...] = (),
) -> None:
    '''Registra `tcdm.<tabla>` sobre los Parquet de `ruta`, con el esquema que infiere Spark.

    Todo `timestamp_ntz` se declara `timestamp`, el único tipo de fecha con hora que
    entienden Spark y Trino. Si hay columnas de partición, se descubren sus directorios.
    '''
    comentarios = comentarios or {}
    inferido: StructType = spark.read.parquet(ruta).schema
    declarado: StructType = StructType(
        [
            StructField(
                campo.name,
                TimestampType() if isinstance(campo.dataType, TimestampNTZType) else campo.dataType,
                campo.nullable,
                {"comment": comentarios[campo.name]} if campo.name in comentarios else {},
            )
            for campo in inferido.fields
        ]
    )
    sentencia: str = f"CREATE TABLE tcdm.{tabla} ({declarado.toDDL()}) USING parquet"
    if particiones:
        sentencia += f" PARTITIONED BY ({', '.join(particiones)})"
    if descripcion:
        sentencia += f" COMMENT '{literal(descripcion)}'"
    spark.sql(f"DROP TABLE IF EXISTS tcdm.{tabla}")
    spark.sql(f"{sentencia} LOCATION '{ruta}'")
    if particiones:
        spark.sql(f"MSCK REPAIR TABLE tcdm.{tabla}")


tablas_raw: list[str] = [
    ruta.rsplit("/", 1)[-1]
    for ruta in listar_directorios(f"{RAW_TPCDS}/*")
    if not ruta.endswith("/item")
]
for nombre in tablas_raw:
    descripciones: dict[str, str] = {c.name: c.description for c in apendice[nombre].columns}
    registrar(
        f"raw_{nombre}",
        f"{HDFS_RAIZ}{RAW_TPCDS}/{nombre}",
        apendice[nombre].description,
        descripciones,
    )
print(f"Registradas {len(tablas_raw)} tablas de raw en el metastore.")
spark.sql("SHOW TABLES IN tcdm LIKE 'raw_*'").show(30)

Spark ha escrito esas definiciones en el metastore. Falta comprobar lo que
decía la regla del principio: que **Trino lee igual** lo que Spark ha
registrado. Contamos las filas de cada tabla con los dos motores y las
comparamos.

In [ ]:
filas_por_tabla: list[dict[str, str | int]] = []
for nombre in tablas_raw:
    filas_spark: int = spark.table(f"tcdm.raw_{nombre}").count()
    filas_trino: int = int(
        run_trino(f"SELECT count(*) AS filas FROM hive.tcdm.raw_{nombre}").iloc[0, 0]
    )
    filas_por_tabla.append(
        {"tabla": f"raw_{nombre}", "filas en Spark": filas_spark, "filas en Trino": filas_trino}
    )

comparacion_raw: pd.DataFrame = pd.DataFrame(filas_por_tabla)
assert (comparacion_raw["filas en Spark"] == comparacion_raw["filas en Trino"]).all()
comparacion_raw

Los dos motores cuentan lo mismo en las 17 tablas. Quedaba la promesa sobre las
fechas con hora: `raw_web_sales` tiene `ws_sold_timestamp`, y la primera y la
última venta deben ser las mismas vistas desde Spark y desde Trino.

In [ ]:
extremos_sql: str = (
    "SELECT min(ws_sold_timestamp) AS primera, max(ws_sold_timestamp) AS ultima FROM {}"
)
pd.concat(
    [
        spark.sql(extremos_sql.format("tcdm.raw_web_sales")).toPandas().assign(motor="Spark"),
        run_trino(extremos_sql.format("hive.tcdm.raw_web_sales")).assign(motor="Trino"),
    ],
    ignore_index=True,
)

### Y `silver`, que escribió Spark

`silver` es una sola tabla, `web_sales_silver`, y la registraste con una DDL de
Trino con las columnas escritas. Sus Parquet los escribió Spark en S5, que ya
declara `ws_sold_timestamp` como `timestamp`, así que Spark puede registrarlos
infiriendo el esquema, con sus columnas de partición, y Trino los lee. Lo
comprobamos sobre los mismos ficheros, con una tabla de prueba que borramos al
terminar. Usa la misma función `registrar`, que fuerza el tipo por si esa
`silver` viniera de una versión anterior de S5, que escribía `timestamp_ntz`.

In [ ]:
registrar(
    "silver_spark_demo",
    f"{HDFS_RAIZ}/datalake/silver/tpcds/web_sales",
    particiones=("sold_year", "sold_month"),
)
filas_silver: dict[str, int] = {
    "web_sales_silver, DDL de Trino, desde Trino": int(
        run_trino("SELECT count(*) FROM hive.tcdm.web_sales_silver").iloc[0, 0]
    ),
    "silver_spark_demo, Spark infiere, desde Trino": int(
        run_trino("SELECT count(*) FROM hive.tcdm.silver_spark_demo").iloc[0, 0]
    ),
    "silver_spark_demo, Spark infiere, desde Spark": spark.table("tcdm.silver_spark_demo").count(),
}
spark.sql("DROP TABLE IF EXISTS tcdm.silver_spark_demo")
assert len(set(filas_silver.values())) == 1, filas_silver
pd.Series(filas_silver, name="filas").to_frame()

Los tres recuentos coinciden: Spark y Trino se entienden sobre `silver` tanto si
la DDL la escribe Trino como si la infiere Spark. `web_sales_silver` se queda
con la DDL de Trino porque así se ve en esta sesión un `CREATE TABLE` con
columnas, con `partitioned_by` y con el descubrimiento de particiones de
Trino; no por una limitación.

La regla queda así: se infiere siempre que se pueda, **se fuerza `timestamp`
donde Spark infiera `timestamp_ntz`**, y se comprueba que el otro motor lee lo
registrado. Registrar con un motor es cómodo mientras los dos entiendan los
mismos tipos.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #a54467;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a54467" size="8" noshade style="display:none"><div style="color:#a54467;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a54467" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Todo <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>raw</code> en el catálogo</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>17 tablas registradas con Spark, que infiere las columnas de los Parquet</li><li style='margin:10px 0;list-style-position:outside'>Comentarios de tabla y de columna tomados del apéndice de S2</li><li style='margin:10px 0;list-style-position:outside'>Trino lee lo que Spark registra, mientras los tipos existan en los dos</li><li style='margin:10px 0;list-style-position:outside'>Las fechas con hora se registran como <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>timestamp</code>, no <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>timestamp_ntz</code>, que Trino no entiende; también <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>silver</code></li></ul><hr color="#a54467" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Buscar en el catálogo</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Con Trino: <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>information_schema</code>, por nombre y por significado</li><li style='margin:10px 0;list-style-position:outside'>Con Spark: un inventario del catálogo y sus ubicaciones</li><li style='margin:10px 0;list-style-position:outside'>Qué hay en el data lake que el catálogo no conoce</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #0c304d;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#0c304d" size="8" noshade style="display:none"><div style="color:#0c304d;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#0c304d" style="color:inherit"><b>Pregunta guía</b></font></div><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>¿En qué tablas aparecerá una columna con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>customer</code> en el nombre?</li><li style='margin:10px 0;list-style-position:outside'>¿Qué tablas de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>tcdm</code> crees que no tienen comentario?</li></ul><hr color="#0c304d" size="2" noshade style="display:none"></div>

## Buscar en el catálogo con Trino

Cada catálogo de Trino tiene su propio `information_schema`, con las vistas
estándar de SQL `tables` y `columns`. Son metadatos del metastore servidos
como tablas, y se consultan con SQL normal: `hive.information_schema.columns`
tiene una fila por cada columna de cada tabla de `hive`, con su tipo y su
comentario. Además, `system.metadata.table_comments` da el comentario de cada
tabla.

Primero, por nombre: ¿dónde aparece algo del cliente? Los nombres de TPC-DS
llevan un prefijo por tabla (`ws_`, `c_`...), por eso filtramos con
`LIKE '%customer%'`.

In [ ]:
run_trino('''
    SELECT table_name, column_name, data_type
    FROM hive.information_schema.columns
    WHERE table_schema = 'tcdm' AND column_name LIKE '%customer%'
    ORDER BY table_name, ordinal_position
''')

Ahora por significado. El nombre no siempre dice de qué trata una columna,
pero el comentario que le pusimos al registrarla sí: buscamos las columnas
cuya descripción habla de descuentos.

In [ ]:
run_trino('''
    SELECT table_name, column_name, comment
    FROM hive.information_schema.columns
    WHERE table_schema = 'tcdm' AND lower(comment) LIKE '%descuento%'
    ORDER BY table_name, ordinal_position
''')

Por último, una auditoría: las tablas que no tienen comentario. Es la
práctica de «cada tabla con su descripción» convertida en una consulta que
se puede repetir.

In [ ]:
run_trino('''
    SELECT table_name
    FROM system.metadata.table_comments
    WHERE catalog_name = 'hive' AND schema_name = 'tcdm' AND comment IS NULL
    ORDER BY table_name
''')

`information_schema` tiene dos límites. Es de un solo catálogo: las tablas
Iceberg de S7 estarán en `iceberg.information_schema`, no en éste. Y no
guarda la ubicación ni las propiedades de la tabla: eso sólo se obtiene con
`SHOW CREATE TABLE` o con `"<tabla>$properties"`, tabla a tabla. Para buscar
por ubicación o por linaje usamos Spark.

## Buscar con Spark: un inventario del catálogo

Spark no ofrece un `information_schema` sobre el metastore de Hive. Lo que
tiene es la API `spark.catalog`: `listTables` y `listColumns` devuelven las
tablas y las columnas, y `DESCRIBE TABLE EXTENDED` la ubicación (es lo que ya
usa `leer_linaje`). Con ellas montamos un inventario, un DataFrame con una
fila por columna, lo registramos como vista temporal y lo consultamos con SQL:
el catálogo, convertido en datos como cualquier otro.

In [ ]:
from pyspark.sql import DataFrame

type FilaInventario = tuple[str, str, str | None, str, str, str, bool, str | None]


def construir_inventario(base_de_datos: str) -> DataFrame:
    '''Una fila por columna de cada tabla de la base de datos, con los datos de su tabla.'''
    del_catalogo: dict[str, TablaConLinaje] = leer_linaje(base_de_datos)
    filas: list[FilaInventario] = []
    for tabla in spark.catalog.listTables(base_de_datos):
        if tabla.isTemporary:
            continue
        nombre: str = f"{base_de_datos}.{tabla.name}"
        ubicacion: str = del_catalogo[nombre].ubicacion if nombre in del_catalogo else ""
        for columna in spark.catalog.listColumns(nombre):
            filas.append(
                (
                    tabla.name,
                    tabla.tableType,
                    tabla.description,
                    ubicacion,
                    columna.name,
                    columna.dataType,
                    columna.isPartition,
                    columna.description,
                )
            )
    return spark.createDataFrame(
        filas,
        schema=(
            "tabla string, tipo_de_tabla string, descripcion_tabla string, ubicacion string, "
            "columna string, tipo string, es_particion boolean, comentario string"
        ),
    )


inventario: DataFrame = construir_inventario("tcdm")
inventario.createOrReplaceTempView("inventario")
print(f"{inventario.count()} columnas en {inventario.select('tabla').distinct().count()} tablas.")

Primera búsqueda, por ubicación: cuántas tablas y columnas hay en cada capa
del data lake y en el warehouse.

In [ ]:
spark.sql('''
    SELECT regexp_extract(ubicacion, '^/(datalake/[a-z]+|warehouse)', 1) AS capa,
           count(DISTINCT tabla) AS tablas,
           count(*) AS columnas
    FROM inventario
    GROUP BY 1
    ORDER BY 1
''').toPandas()

Segunda, entre tablas: qué columnas aparecen en más de una. Es la forma de
seguir una columna por las capas, de `raw` a `silver` y a `gold`.

In [ ]:
spark.sql('''
    SELECT columna, count(*) AS tablas, array_sort(collect_list(tabla)) AS en_las_tablas
    FROM inventario
    GROUP BY columna
    HAVING count(*) > 1
    ORDER BY tablas DESC, columna
''').toPandas()

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #0c304d;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#0c304d" size="8" noshade style="display:none"><div style="color:#0c304d;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#0c304d" style="color:inherit"><b>Pregunta guía</b></font></div><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>¿Qué directorios de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>/datalake</code> quedarán sin tabla en el catálogo?</li><li style='margin:10px 0;list-style-position:outside'>¿Por qué ése, y no ninguno de los de TPC-DS?</li></ul><hr color="#0c304d" size="2" noshade style="display:none"></div>

Una búsqueda más, la inversa: no qué hay en el catálogo, sino qué hay en el
data lake que **el catálogo no conoce**. Comparamos los directorios de
HDFS, `/datalake/<capa>/<conjunto>/<tabla>`, con las ubicaciones del
inventario.

In [ ]:
en_hdfs: set[str] = set(listar_directorios("/datalake/*/*/*"))
registradas: set[str] = {
    fila["ubicacion"] for fila in inventario.select("ubicacion").distinct().collect()
}
pd.DataFrame({"directorio sin tabla": sorted(en_hdfs - registradas)})

Queda `/datalake/raw/ign/municipios`: son los municipios del IGN que S2 dejó
como CSV comprimido, no como Parquet. Spark podría registrarlo con
`USING csv` y sus opciones (separador, cabecera, codificación), pero Trino no
vería sus columnas, así que no sería una tabla compartida. Por eso queda fuera
hasta que se convierta a Parquet.

Las dos herramientas se reparten el trabajo. **Trino** busca por nombre y por
significado con SQL estándar, sobre un catálogo. **Spark** busca por ubicación
y por linaje, y deja el inventario en un DataFrame que se puede cruzar con
cualquier otro dato. Las dos leen del mismo metastore.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #a54467;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a54467" size="8" noshade style="display:none"><div style="color:#a54467;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a54467" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Buscar en el catálogo</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Trino: <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>information_schema.columns</code> y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>system.metadata.table_comments</code>, por nombre y por comentario</li><li style='margin:10px 0;list-style-position:outside'>Spark: <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>spark.catalog</code> y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>DESCRIBE TABLE EXTENDED</code>, convertidos en un inventario con ubicación</li><li style='margin:10px 0;list-style-position:outside'>Contrastar el catálogo con HDFS deja ver lo que no está registrado</li><li style='margin:10px 0;list-style-position:outside'>Sin comentarios y sin registro no hay nada que buscar: documentar es parte de registrar</li></ul><hr color="#a54467" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #2f6f4f;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#2f6f4f" size="8" noshade style="display:none"><div style="color:#2f6f4f;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#2f6f4f" style="color:inherit"><b>Actividades</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Busca en tu catálogo</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Con Trino, busca las columnas cuyo comentario hable de devoluciones y las tablas que las contienen</li><li style='margin:10px 0;list-style-position:outside'>Añade con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>COMMENT ON TABLE</code> una descripción a las tablas que la auditoría señala y repite la auditoría</li><li style='margin:10px 0;list-style-position:outside'>Con Spark, busca las tablas con más columnas y las que tienen columnas <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>decimal</code></li><li style='margin:10px 0;list-style-position:outside'>Cuenta con el inventario cuántas columnas tiene cada tabla de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>raw</code> y compáralo con el apéndice de S2</li></ul><hr color="#2f6f4f" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Buenas prácticas de catálogo</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Cada tabla con propietario, descripción, ubicación y contrato de esquema y claves</li><li style='margin:10px 0;list-style-position:outside'>Particiones justificadas por las consultas; estadísticas generadas y revisadas cuando hacen falta</li><li style='margin:10px 0;list-style-position:outside'>Fuente y derivadas tienen ciclos de vida distintos: borrar una derivada nunca borra su fuente</li><li style='margin:10px 0;list-style-position:outside'>Cada derivada apunta su linaje: de qué sale y qué proceso la escribe</li><li style='margin:10px 0;list-style-position:outside'>La limpieza va separada de las consultas normales</li><li style='margin:10px 0;list-style-position:outside'>El metastore no da transacciones ni versiones: Iceberg las añade sobre una tabla administrada en S7</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Buenas prácticas de catálogo y data lake

El metastore permite formalizar las decisiones de S5:

- cada tabla debería tener un propietario y una descripción; este notebook
  la demuestra con `item` y con las tablas de `raw`, y la auditoría de
  comentarios señala las que faltan;
- la ubicación física está documentada;
- el esquema y las claves tienen un contrato;
- las columnas de partición se justifican por las consultas que se van a
  hacer, no se añaden "por si acaso";
- los ficheros auxiliares (`_SUCCESS`, *checksums*...) no se confunden con
  datos de la tabla;
- las estadísticas se generan y se revisan cuando son necesarias, no una vez
  y para siempre;
- las tablas fuente y las tablas derivadas tienen ciclos de vida distintos:
  borrar una tabla derivada nunca debe poder borrar su fuente;
- cada tabla derivada apunta su linaje: de qué tablas o rutas se calcula y
  qué proceso la escribe;
- una regeneración no borra datos ajenos;
- las operaciones de limpieza están separadas de las consultas normales, como
  se hace en la última sección de este notebook.

El metastore mejora la localización y la reutilización, pero no proporciona
por sí solo transacciones completas, historial de versiones ni una política
de calidad de datos. Un `INSERT` fallido a mitad de camino, dos escrituras
concurrentes sobre la misma partición o la necesidad de "viajar en el
tiempo" a una versión anterior de la tabla son problemas que el Hive
Metastore no resuelve. Esas limitaciones son exactamente las que motivan
introducir Iceberg en S7.

Ninguna depende de dónde viven los ficheros, sino de quién lleva la cuenta de
ellos: una tabla Hive, externa o administrada, sólo registra un directorio.
Iceberg conserva el catálogo y la zona `/warehouse`, pero añade a la tabla
administrada metadatos propios que registran cada escritura como una versión
nueva. De ahí salen las consultas ACID sobre una tabla cuyos datos siguen
siendo Parquet en HDFS, legibles por otras herramientas, pero cuya gestión
corresponde por entero al metastore y a Iceberg. En S7, `tcdm.web_sales`
pasa de ser una fuente externa a ser una tabla administrada Iceberg.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Explicar una decisión</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Escoge una de las preguntas y localiza la celda que permite responderla.</li><li style='margin:10px 0;list-style-position:outside'>Usa su salida para justificar tu respuesta.</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Preguntas para interpretar la experiencia

- Si `web_sales_silver` tuviera miles de particiones, ¿qué tendría que
  hacer un motor sin catálogo para planificar una consulta filtrada por
  `sold_month`? ¿Qué hace en su lugar Trino con el metastore?
- ¿Qué diferencia observable hay entre `tcdm.item` y
  `tcdm.web_sales_by_year_managed_demo` en cuanto a lo que ocurre en HDFS al
  ejecutar el `CREATE TABLE`?
- ¿Por qué `SELECT count(*) FROM hive.tcdm.web_sales_silver` devolvió 0 filas
  justo después del `CREATE TABLE`, si los ficheros ya existían en HDFS
  desde que terminó S5?
- ¿Por qué Trino no admite `MSCK REPAIR TABLE` y qué usa en su lugar? ¿Por
  qué esa misma sentencia sí funciona en Spark SQL?
- ¿Qué información añade `hive.tcdm."web_sales_silver$partitions"` que no
  aparece en un `DESCRIBE` normal de la tabla?
- ¿Qué cambia entre el plan de `EXPLAIN` de una consulta sin filtrar y el de
  una consulta filtrada por `sold_year`/`sold_month`? ¿Dónde exactamente
  aparece la poda de particiones en el plan?
- ¿Qué mostraba `SHOW STATS FOR hive.tcdm.web_sales_silver` antes de
  `ANALYZE` y qué cambió después? ¿Para qué las usa el planificador?
- ¿Por qué se dice que Spark y Trino son "dos clientes de la misma
  definición" y no "dos catálogos distintos"? ¿Qué comprobación de esta
  sesión lo demuestra?
- ¿Qué podría pasar si, tras crear una tabla externa con `CREATE TABLE`,
  ejecutas `DROP TABLE` sobre ella? ¿Y si en su lugar borraras la tabla
  administrada `tcdm.web_sales_by_year_managed_demo`? ¿Por qué la respuesta
  es distinta en cada caso?
- ¿Qué tablas habría que recalcular, y en qué orden, si llegaran ventas
  nuevas a `/datalake/raw/tpcds/web_sales`? ¿Por qué `tcdm.item` no se
  recalcula nunca? ¿Qué respondería la lista si alguien cambiara el proceso
  de S5 sin actualizar las propiedades de linaje?
- ¿Qué ventaja tiene que una tabla administrada tenga un único dueño
  (catálogo y motor) frente a una tabla externa sobre `/datalake`? ¿Por qué
  la tabla administrada de esta sesión no basta para tener consultas ACID, y
  qué añadirá Iceberg sobre esa misma ubicación?
- ¿Por qué `raw` se registra con Spark y `web_sales_silver` con Trino? ¿Qué
  comprobación hay que hacer siempre que un motor registra tablas que va a
  leer otro?
- ¿Qué buscarías con `information_schema` de Trino y qué con el inventario
  de Spark? ¿Qué directorio de `/datalake` no está en el catálogo, y por qué?
- ¿Qué limitación del Hive Metastore, de las mencionadas en «Buenas
  prácticas de catálogo y data lake», esperarías que resolviera un formato
  de tabla como Iceberg?

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Preparar la demostración</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Elige una salida del notebook que muestre un cambio medible.</li><li style='margin:10px 0;list-style-position:outside'>Explica qué hiciste, qué esperabas y qué observaste.</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Evidencias para la siguiente sesión

Antes de la siguiente sesión tendrás una reunión individual breve con el
profesor para revisar el trabajo de esta sesión. Esa reunión combina una
demostración en vivo sobre tu propio ordenador y una memoria escrita breve.

### Qué mostrar en el ordenador durante la reunión

Ten preparadas estas evidencias. El «antes» de la 2 y de la 5 sólo existe una vez
por tabla: enseña las salidas que quedaron guardadas en el notebook o, para
repetir el paso de 0 a N filas en vivo, vuelve a ejecutar desde «Registrar la
versión particionada», que borra y recrea la definición.

1. El `SHOW CREATE TABLE` de `tcdm.item`, `tcdm.web_sales_silver` y
   `tcdm.web_sales_by_year_gold`, mostrando que las tres son tablas externas
   con `external_location` apuntando a `/datalake`.
2. El recuento de filas de `tcdm.web_sales_silver` antes y después de
   `CALL hive.system.sync_partition_metadata(...)`, junto con la explicación
   de por qué cambia.
3. El contenido de `hive.tcdm."web_sales_silver$partitions"`.
4. Un plan de `EXPLAIN` (o `EXPLAIN ANALYZE`) que muestre la poda de
   particiones al filtrar por `sold_year`/`sold_month`.
5. La tabla de `SHOW STATS FOR hive.tcdm.web_sales_silver` antes y después de
   `ANALYZE`.
6. El mismo resultado de negocio de `tcdm.web_sales_by_year_gold` obtenido
   desde Trino y desde Spark, comprobando que coinciden fila a fila.
7. El `SHOW CREATE TABLE` de `tcdm.web_sales_by_year_managed_demo` y el
   listado HDFS de su ubicación bajo `/warehouse/tcdm.db/`, comparado con el
   de una tabla externa.
8. La propiedad `tcdm.linaje.origenes` de `tcdm.web_sales_by_year_gold`
   leída desde Trino, y la lista de tablas que habría que recalcular si
   cambiara `/datalake/raw/tpcds/web_sales`, en su orden.
9. Que sabes explicar, con tus propias palabras, la diferencia entre
   eliminar la definición de una tabla externa y eliminar una tabla
   administrada, y por qué Iceberg se apoya en una tabla administrada.
10. Una búsqueda en el catálogo con Trino (por columna o por comentario) y
    otra con el inventario de Spark (por ubicación), y el directorio de
    `/datalake` que no tiene tabla.
11. Tu tabla administrada propia y tu tabla derivada con su linaje, con sus
    observaciones de los ejercicios.

### Memoria escrita (una o dos páginas)

Trae también un documento breve —una o dos páginas, no hace falta más— que
no se limite a pegar capturas de las salidas anteriores: debe explicar con
tus propias palabras qué papel tiene cada pieza de la arquitectura
(PostgreSQL, Hive Metastore, Trino y Spark); qué diferencia hay entre una
tabla externa y una administrada, qué ocurre con los datos al borrar cada
una y qué ventaja tiene la administrada de cara a Iceberg; por qué la tabla particionada devolvía cero filas hasta sincronizar
sus particiones; qué es el linaje de una tabla y cómo lo has apuntado; para qué sirven las estadísticas de `ANALYZE`; y qué
limitaciones del metastore justifican introducir Iceberg.

### Qué es importante de cara al examen final

El examen no pide recordar la sintaxis exacta de una sentencia. Debes poder
explicar:

- qué problema resuelve un catálogo frente a una ruta física, también con
  miles de particiones, y qué guarda y qué no guarda el Hive Metastore;
- qué es el linaje de una tabla, dónde se puede guardar y cómo responde a
  la pregunta de qué recalcular cuando cambia una fuente;
- por qué el metastore no es un motor de consultas y Trino no es un
  catálogo, y qué significa que Spark y Trino sean dos clientes de la misma
  definición de tabla;
- la diferencia entre tabla externa y tabla administrada: ubicación y
  ciclo de vida de los datos;
- por qué Iceberg se construye sobre una tabla administrada: un único dueño
  de los ficheros permite escrituras atómicas y consultas ACID, mientras los
  datos siguen siendo Parquet;
- que una partición Hive son dos cosas, un directorio `columna=valor` y una
  fila de metadatos, y por qué hace falta un paso de descubrimiento;
- cómo se reconoce en un plan la poda de particiones y la selección de
  columnas;
- qué son las estadísticas de tabla y de columna y para qué las usa el
  planificador;
- qué no ofrece el metastore: transacciones, historial de versiones y
  evolución segura del esquema.

### Cerrar la sesión de Spark

Cierra la sesión Spark sobre YARN al terminar los cálculos. Así se liberan los contenedores de YARN y memoria en `namenode` y queda disponible el puerto 4040 para la aplicación de la sesión siguiente.


In [ ]:
spark.stop()

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Decidir qué conservar</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Localiza los datos y metadatos que necesitarás al volver al laboratorio.</li><li style='margin:10px 0;list-style-position:outside'>Lee las consecuencias de cada orden antes de elegirla.</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Detener y limpiar

Estas operaciones se muestran como texto, no como celdas de código, para que
un «ejecutar todo» de este notebook no las dispare por accidente. La tabla
administrada de esta sesión es un recurso de laboratorio: consérvala hasta
la reunión de evidencias, que la necesita, y bórrala después. **Nunca**
apliques el mismo razonamiento a las tablas externas, cuyo `DROP TABLE`
sólo afecta al catálogo; y no elimines sus definiciones antes de terminar
S7, que parte de ellas.

Las órdenes `docker compose` de la tabla se ejecutan en una terminal de tu
equipo, desde la raíz de la distribución; los `DROP TABLE` se ejecutan en
el cliente Trino indicado en cada caso.

| Situación | Orden | Consecuencia |
| --- | --- | --- |
| Eliminar sólo la tabla de laboratorio (administrada) | `DROP TABLE hive.tcdm.web_sales_by_year_managed_demo` (desde el cliente Trino de este notebook, o `docker exec trino-hdfs trino --user luser --execute "..."` desde una terminal) | Borra tanto la entrada del metastore como los ficheros Parquet bajo `/warehouse/tcdm.db/web_sales_by_year_managed_demo`: es una tabla administrada, su ciclo de vida de datos depende del catálogo. |
| Eliminar las definiciones externas de esta sesión | `DROP TABLE hive.tcdm.item`, `DROP TABLE hive.tcdm.web_sales_silver`, `DROP TABLE hive.tcdm.web_sales_by_year_gold` | Borra únicamente las tres entradas del metastore. Los Parquet de `/datalake/raw`, `/datalake/silver` y `/datalake/gold` siguen intactos: nunca interpretes este `DROP TABLE` como "borrar el data lake". S7 necesita las tres definiciones: no las elimines hasta haberla terminado. |
| Parar temporalmente el warehouse | `docker compose -f entorno/compose-warehouse-hdfs.yml stop` | Conserva PostgreSQL, Hive Metastore y Trino (y su contenido); se reanuda con `start`. No afecta al clúster Hadoop. |
| Eliminar el warehouse por completo (destructivo) | `docker compose -f entorno/compose-warehouse-hdfs.yml down -v --remove-orphans` | Borra también el volumen de PostgreSQL y las definiciones del metastore, pero no `/datalake` ni `/warehouse` en HDFS. Si la tabla administrada de laboratorio seguía creada, sus ficheros quedan huérfanos bajo `/warehouse/tcdm.db/web_sales_by_year_managed_demo`; antes de repetir su CTAS, bórralos con `!hdfs dfs -rm -r -f /warehouse/tcdm.db/web_sales_by_year_managed_demo` en una celda del kernel de `namenode`. |

No confundas ese `down -v` del warehouse con el `down`/`volume rm` de S1
(clúster Hadoop) o con el `down -v` de la sesión 3 (RustFS): cada Compose
tiene su propio
volumen y afecta sólo a sus propios servicios.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #a54467;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a54467" size="8" noshade style="display:none"><div style="color:#a54467;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a54467" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Sesión 6</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Nombres, esquemas y particiones compartidos entre Trino y Spark</li><li style='margin:10px 0;list-style-position:outside'>El linaje en las propiedades de tabla: qué recalcular cuando cambia <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>raw</code></li><li style='margin:10px 0;list-style-position:outside'>Todo <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>raw</code> registrado con Spark y comentado, y búsquedas por nombre, comentario y ubicación</li><li style='margin:10px 0;list-style-position:outside'>Sin transacciones ni versiones sobre Parquet: Iceberg las añade a una tabla administrada del warehouse, y con ellas ACID</li><li style='margin:10px 0;list-style-position:outside'>S7 construye sobre esta misma tabla <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>web_sales_silver</code></li></ul><hr color="#a54467" size="2" noshade style="display:none"></div>

## Siguiente paso

S6 muestra que el metastore da nombres y metadatos compartidos a una
organización de Parquet, y que Spark y Trino pueden consultar exactamente la
misma definición sin duplicar configuración. S7 introducirá Iceberg como
formato de tabla para gestionar cambios, snapshots y particionado evolutivo,
partiendo de la misma `tcdm.web_sales_silver` que se acaba de registrar aquí.